In [1]:
import sys
import pandas as pd
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch_geometric

print("Python executable:")
print(sys.executable)

print("\nVersions:")
print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("pyarrow:", pa.__version__)
print("pyarrow.parquet OK:", pq is not None)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("torch_geometric:", torch_geometric.__version__)

print("\nCUDA:")
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Python executable:
E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe

Versions:
pandas: 2.3.3
geopandas: 1.1.3
pyarrow: 24.0.0
pyarrow.parquet OK: True
torch: 2.12.0+cu126
torch cuda: 12.6
torch_geometric: 2.7.0

CUDA:
cuda available: True
gpu: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
# ============================================================
# FREIGHT-MNet Processing Step 1:
# Census County Boundaries -> GeoParquet + County Master
#
# Input:
#   E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\extracted
#
# Output:
#   - county_cartographic_500k_2024.parquet
#   - county_tigerline_2024.parquet
#   - county_master_2024.parquet
#   - county_boundaries_master_2024.parquet
#   - east_plus_gulf_counties_2024.parquet
#
# East-Plus-Gulf definition:
#   CONUS counties whose centroid longitude >= -100
# ============================================================

from pathlib import Path
import json
import zipfile
import traceback

import pandas as pd
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
CENSUS_ROOT = DATA_ROOT / "07_census_boundaries" / "counties_2024"

RAW_ZIP_DIR = CENSUS_ROOT / "raw_zip"
EXTRACTED_DIR = CENSUS_ROOT / "extracted"
GEOPARQUET_DIR = CENSUS_ROOT / "geoparquet"
PROCESSED_DIR = CENSUS_ROOT / "processed"
METADATA_DIR = CENSUS_ROOT / "_metadata"
PREVIEW_DIR = CENSUS_ROOT / "preview"

MANIFEST_DIR = DATA_ROOT / "00_manifest"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

for d in [GEOPARQUET_DIR, PROCESSED_DIR, METADATA_DIR, PREVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EAST_PLUS_GULF_LONGITUDE_THRESHOLD = -100.0

print("=" * 100)
print("Census county boundary processing")
print("DATA_ROOT:", DATA_ROOT)
print("CENSUS_ROOT:", CENSUS_ROOT)
print("pyarrow:", pa.__version__)
print("pyarrow.parquet OK:", pq is not None)
print("=" * 100)

# ------------------------------------------------------------
# 1. Source definitions
# ------------------------------------------------------------

SOURCES = {
    "cartographic_500k": {
        "zip_path": RAW_ZIP_DIR / "cb_2024_us_county_500k.zip",
        "extract_dir": EXTRACTED_DIR / "cartographic_500k",
        "expected_shp": "cb_2024_us_county_500k.shp",
        "parquet_path": GEOPARQUET_DIR / "county_cartographic_500k_2024.parquet",
    },
    "tigerline": {
        "zip_path": RAW_ZIP_DIR / "tl_2024_us_county.zip",
        "extract_dir": EXTRACTED_DIR / "tigerline",
        "expected_shp": "tl_2024_us_county.shp",
        "parquet_path": GEOPARQUET_DIR / "county_tigerline_2024.parquet",
    },
}

MASTER_SOURCE_KEY = "cartographic_500k"

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def extract_if_needed(zip_path: Path, extract_dir: Path) -> None:
    extract_dir.mkdir(parents=True, exist_ok=True)
    shp_files = list(extract_dir.glob("*.shp"))

    if shp_files:
        print(f"[skip extraction] found shapefile in {extract_dir}")
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Missing ZIP and no extracted shapefile found: {zip_path}")

    print(f"[extract] {zip_path} -> {extract_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    shp_files = list(extract_dir.glob("*.shp"))
    if not shp_files:
        raise RuntimeError(f"No shapefile found after extraction: {extract_dir}")


def find_shapefile(extract_dir: Path, expected_name: str) -> Path:
    expected = extract_dir / expected_name
    if expected.exists():
        return expected

    shp_files = sorted(extract_dir.glob("*.shp"))
    if not shp_files:
        raise FileNotFoundError(f"No .shp file found in {extract_dir}")

    return shp_files[0]


def write_source_geoparquet(source_key: str, cfg: dict) -> dict:
    extract_if_needed(cfg["zip_path"], cfg["extract_dir"])
    shp_path = find_shapefile(cfg["extract_dir"], cfg["expected_shp"])
    parquet_path = cfg["parquet_path"]

    print("\n" + "=" * 100)
    print("SOURCE:", source_key)
    print("SHP:", shp_path)
    print("PARQUET:", parquet_path)
    print("=" * 100)

    gdf = gpd.read_file(shp_path)

    if gdf.crs is None:
        # Census county files are normally NAD83 / EPSG:4269.
        gdf = gdf.set_crs("EPSG:4269")

    gdf = gdf.to_crs("EPSG:4326")

    # Ensure GEOID exists.
    if "GEOID" not in gdf.columns:
        if {"STATEFP", "COUNTYFP"}.issubset(gdf.columns):
            gdf["GEOID"] = (
                gdf["STATEFP"].astype(str).str.zfill(2)
                + gdf["COUNTYFP"].astype(str).str.zfill(3)
            )
        else:
            raise RuntimeError(f"Cannot construct GEOID. Columns: {list(gdf.columns)}")

    # Normalize identifiers.
    gdf["STATEFP"] = gdf["STATEFP"].astype(str).str.zfill(2)
    gdf["COUNTYFP"] = gdf["COUNTYFP"].astype(str).str.zfill(3)
    gdf["GEOID"] = gdf["GEOID"].astype(str).str.zfill(5)

    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_parquet(parquet_path, index=False)

    # Preview and fields.
    preview_path = PREVIEW_DIR / f"{source_key}_preview_first20.csv"
    gdf.drop(columns="geometry").head(20).to_csv(preview_path, index=False, encoding="utf-8-sig")

    fields_path = METADATA_DIR / f"{source_key}_fields.csv"
    pd.DataFrame({
        "column": list(gdf.columns),
        "dtype": [str(t) for t in gdf.dtypes],
    }).to_csv(fields_path, index=False, encoding="utf-8-sig")

    result = {
        "source_key": source_key,
        "shp_path": str(shp_path),
        "parquet_path": str(parquet_path),
        "preview_path": str(preview_path),
        "fields_path": str(fields_path),
        "n_rows": len(gdf),
        "crs": str(gdf.crs),
        "n_columns": len(gdf.columns),
        "columns": "|".join(gdf.columns),
        "parquet_size_mb": round(parquet_path.stat().st_size / 1024**2, 4),
    }

    print("Rows:", result["n_rows"])
    print("CRS:", result["crs"])
    print("Columns:", list(gdf.columns))
    print("Saved:", parquet_path)

    return result


def state_fips_name_mapping() -> dict:
    return {
        "01": "Alabama", "02": "Alaska", "04": "Arizona", "05": "Arkansas",
        "06": "California", "08": "Colorado", "09": "Connecticut", "10": "Delaware",
        "11": "District of Columbia", "12": "Florida", "13": "Georgia", "15": "Hawaii",
        "16": "Idaho", "17": "Illinois", "18": "Indiana", "19": "Iowa",
        "20": "Kansas", "21": "Kentucky", "22": "Louisiana", "23": "Maine",
        "24": "Maryland", "25": "Massachusetts", "26": "Michigan", "27": "Minnesota",
        "28": "Mississippi", "29": "Missouri", "30": "Montana", "31": "Nebraska",
        "32": "Nevada", "33": "New Hampshire", "34": "New Jersey", "35": "New Mexico",
        "36": "New York", "37": "North Carolina", "38": "North Dakota", "39": "Ohio",
        "40": "Oklahoma", "41": "Oregon", "42": "Pennsylvania", "44": "Rhode Island",
        "45": "South Carolina", "46": "South Dakota", "47": "Tennessee", "48": "Texas",
        "49": "Utah", "50": "Vermont", "51": "Virginia", "53": "Washington",
        "54": "West Virginia", "55": "Wisconsin", "56": "Wyoming",
        "60": "American Samoa", "66": "Guam", "69": "Northern Mariana Islands",
        "72": "Puerto Rico", "78": "U.S. Virgin Islands",
    }


def build_county_master(master_parquet_path: Path) -> dict:
    print("\n" + "=" * 100)
    print("Building county master from:", master_parquet_path)
    print("=" * 100)

    gdf = gpd.read_parquet(master_parquet_path).to_crs("EPSG:4326")

    required = ["STATEFP", "COUNTYFP", "GEOID", "geometry"]
    missing = [c for c in required if c not in gdf.columns]
    if missing:
        raise RuntimeError(f"Missing required columns {missing}. Columns: {list(gdf.columns)}")

    gdf["STATEFP"] = gdf["STATEFP"].astype(str).str.zfill(2)
    gdf["COUNTYFP"] = gdf["COUNTYFP"].astype(str).str.zfill(3)
    gdf["GEOID"] = gdf["GEOID"].astype(str).str.zfill(5)

    if "NAME" not in gdf.columns:
        gdf["NAME"] = None

    if "NAMELSAD" not in gdf.columns:
        gdf["NAMELSAD"] = gdf["NAME"]

    gdf["state_name"] = gdf["STATEFP"].map(state_fips_name_mapping())

    # Compute centroids.
    # Use CONUS Albers for stable centroid calculation in the main study area.
    gdf_proj = gdf.to_crs("EPSG:5070")

    centroid_proj = gdf_proj.geometry.centroid
    centroid_lonlat = gpd.GeoSeries(centroid_proj, crs="EPSG:5070").to_crs("EPSG:4326")

    gdf["centroid_lon"] = centroid_lonlat.x
    gdf["centroid_lat"] = centroid_lonlat.y

    rep_proj = gdf_proj.geometry.representative_point()
    rep_lonlat = gpd.GeoSeries(rep_proj, crs="EPSG:5070").to_crs("EPSG:4326")

    gdf["rep_lon"] = rep_lonlat.x
    gdf["rep_lat"] = rep_lonlat.y

    # Exclude Alaska, Hawaii, and territories from CONUS mask.
    non_conus_statefps = {"02", "15", "60", "66", "69", "72", "78"}
    gdf["is_conus"] = ~gdf["STATEFP"].isin(non_conus_statefps)

    # East-Plus-Gulf: CONUS counties east of 100°W.
    gdf["east_plus_gulf_flag"] = (
        gdf["is_conus"]
        & (gdf["centroid_lon"] >= EAST_PLUS_GULF_LONGITUDE_THRESHOLD)
    )

    # Table-only county master.
    keep_cols = [
        "GEOID",
        "STATEFP",
        "COUNTYFP",
        "NAME",
        "NAMELSAD",
        "state_name",
        "centroid_lon",
        "centroid_lat",
        "rep_lon",
        "rep_lat",
        "is_conus",
        "east_plus_gulf_flag",
    ]

    for optional in ["ALAND", "AWATER", "LSAD", "CLASSFP", "FUNCSTAT"]:
        if optional in gdf.columns and optional not in keep_cols:
            keep_cols.append(optional)

    county_master = gdf[keep_cols].copy().rename(
        columns={
            "GEOID": "county_fips",
            "STATEFP": "statefp",
            "COUNTYFP": "countyfp",
            "NAME": "county_name",
            "NAMELSAD": "county_namelsad",
        }
    )

    # Geo county boundaries with project-style names.
    county_gdf = gdf.rename(
        columns={
            "GEOID": "county_fips",
            "STATEFP": "statefp",
            "COUNTYFP": "countyfp",
            "NAME": "county_name",
            "NAMELSAD": "county_namelsad",
        }
    )

    # Output paths.
    county_master_csv = PROCESSED_DIR / "county_master_2024.csv"
    county_master_parquet = PROCESSED_DIR / "county_master_2024.parquet"

    county_boundaries_parquet = PROCESSED_DIR / "county_boundaries_master_2024.parquet"
    county_boundaries_geojson = PROCESSED_DIR / "county_boundaries_master_2024.geojson"

    east_counties_csv = PROCESSED_DIR / "east_plus_gulf_counties_2024.csv"
    east_counties_parquet = PROCESSED_DIR / "east_plus_gulf_counties_2024.parquet"
    east_counties_geojson = PROCESSED_DIR / "east_plus_gulf_counties_2024.geojson"

    state_summary_csv = PROCESSED_DIR / "county_state_summary_2024.csv"

    # Save.
    county_master.to_csv(county_master_csv, index=False, encoding="utf-8-sig")
    county_master.to_parquet(county_master_parquet, index=False)

    county_gdf.to_parquet(county_boundaries_parquet, index=False)
    county_gdf.to_file(county_boundaries_geojson, driver="GeoJSON")

    east_gdf = county_gdf[county_gdf["east_plus_gulf_flag"]].copy()
    east_table = county_master[county_master["east_plus_gulf_flag"]].copy()

    east_table.to_csv(east_counties_csv, index=False, encoding="utf-8-sig")
    east_gdf.to_parquet(east_counties_parquet, index=False)
    east_gdf.to_file(east_counties_geojson, driver="GeoJSON")

    state_summary = (
        county_master
        .groupby(["statefp", "state_name", "is_conus"], dropna=False)
        .agg(
            n_counties=("county_fips", "size"),
            n_east_plus_gulf=("east_plus_gulf_flag", "sum"),
            min_centroid_lon=("centroid_lon", "min"),
            max_centroid_lon=("centroid_lon", "max"),
        )
        .reset_index()
        .sort_values("statefp")
    )

    state_summary.to_csv(state_summary_csv, index=False, encoding="utf-8-sig")

    result = {
        "processed_status": "ok",
        "master_source_parquet": str(master_parquet_path),
        "county_master_csv": str(county_master_csv),
        "county_master_parquet": str(county_master_parquet),
        "county_boundaries_master_parquet": str(county_boundaries_parquet),
        "county_boundaries_master_geojson": str(county_boundaries_geojson),
        "east_plus_gulf_counties_csv": str(east_counties_csv),
        "east_plus_gulf_counties_parquet": str(east_counties_parquet),
        "east_plus_gulf_counties_geojson": str(east_counties_geojson),
        "state_summary_csv": str(state_summary_csv),
        "n_counties_total": len(county_master),
        "n_counties_conus": int(county_master["is_conus"].sum()),
        "n_counties_east_plus_gulf": int(county_master["east_plus_gulf_flag"].sum()),
        "longitude_threshold": EAST_PLUS_GULF_LONGITUDE_THRESHOLD,
    }

    return result

# ------------------------------------------------------------
# 3. Run processing
# ------------------------------------------------------------

source_results = []

try:
    for source_key, cfg in SOURCES.items():
        source_results.append(write_source_geoparquet(source_key, cfg))

    master_source_path = SOURCES[MASTER_SOURCE_KEY]["parquet_path"]
    processed_info = build_county_master(master_source_path)

    # Save manifests.
    source_manifest = pd.DataFrame(source_results)
    source_manifest_path = MANIFEST_DIR / "census_county_boundaries_2024_processing_manifest.csv"
    source_manifest.to_csv(source_manifest_path, index=False, encoding="utf-8-sig")

    processed_info_path = MANIFEST_DIR / "census_county_boundaries_2024_processed_outputs.json"
    processed_info_path.write_text(json.dumps(processed_info, indent=2, ensure_ascii=False), encoding="utf-8")

    print("\n" + "=" * 100)
    print("CENSUS COUNTY PROCESSING COMPLETE")
    print("=" * 100)
    print("Source manifest:", source_manifest_path)
    print("Processed outputs JSON:", processed_info_path)

    print("\nCounts:")
    print("  total counties/equivalents:", processed_info["n_counties_total"])
    print("  CONUS counties/equivalents:", processed_info["n_counties_conus"])
    print("  East-Plus-Gulf counties/equivalents:", processed_info["n_counties_east_plus_gulf"])

    print("\nKey outputs:")
    print("  county_master:", processed_info["county_master_parquet"])
    print("  county_boundaries:", processed_info["county_boundaries_master_parquet"])
    print("  east_plus_gulf:", processed_info["east_plus_gulf_counties_parquet"])

    display(source_manifest[["source_key", "n_rows", "crs", "parquet_size_mb", "parquet_path"]])

except Exception as e:
    print("\nFAILED")
    print(repr(e))
    print(traceback.format_exc())
    raise

Census county boundary processing
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
CENSUS_ROOT: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024
pyarrow: 24.0.0
pyarrow.parquet OK: True
[skip extraction] found shapefile in E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\extracted\cartographic_500k

SOURCE: cartographic_500k
SHP: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\extracted\cartographic_500k\cb_2024_us_county_500k.shp
PARQUET: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\geoparquet\county_cartographic_500k_2024.parquet
Rows: 3235
CRS: EPSG:4326
Columns: ['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSAD', 'STUSPS', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'geometry']
Saved: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\geoparquet\county_cartographic_500k_2024.parquet
[skip extraction] found shap

,source_key,n_rows,crs,parquet_size_mb,parquet_path
0,cartographic_500k,3235,EPSG:4326,16.0229,E:\NetworkOptimization\pythonProject1\Data\07_...
1,tigerline,3235,EPSG:4326,120.3448,E:\NetworkOptimization\pythonProject1\Data\07_...


In [3]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
CENSUS_ROOT = DATA_ROOT / "07_census_boundaries" / "counties_2024"
PROCESSED_DIR = CENSUS_ROOT / "processed"

paths = {
    "county_cartographic_parquet": CENSUS_ROOT / "geoparquet" / "county_cartographic_500k_2024.parquet",
    "county_tigerline_parquet": CENSUS_ROOT / "geoparquet" / "county_tigerline_2024.parquet",
    "county_master": PROCESSED_DIR / "county_master_2024.parquet",
    "county_boundaries_master": PROCESSED_DIR / "county_boundaries_master_2024.parquet",
    "east_plus_gulf_counties": PROCESSED_DIR / "east_plus_gulf_counties_2024.parquet",
    "state_summary": PROCESSED_DIR / "county_state_summary_2024.csv",
}

rows = []
for name, p in paths.items():
    rows.append({
        "file": name,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024**2, 4) if p.exists() else 0,
        "path": str(p),
    })

check = pd.DataFrame(rows)
display(check)

county_master = pd.read_parquet(paths["county_master"])
display(county_master.head(20))

print("county_master rows:", len(county_master))
print("\nEast-Plus-Gulf counts:")
display(county_master["east_plus_gulf_flag"].value_counts(dropna=False))

print("\nState summary:")
state_summary = pd.read_csv(paths["state_summary"])
display(state_summary.head(60))

east_gdf = gpd.read_parquet(paths["east_plus_gulf_counties"])
print("\nEast-Plus-Gulf GeoDataFrame rows:", len(east_gdf))
print("CRS:", east_gdf.crs)
print("Geometry types:", dict(east_gdf.geometry.geom_type.value_counts()))

assert east_gdf["is_conus"].all()
assert (east_gdf["centroid_lon"] >= -100.0).all()

print("\nOK: Census county boundaries are processed successfully.")

,file,exists,size_mb,path
0,county_cartographic_parquet,True,16.0229,E:\NetworkOptimization\pythonProject1\Data\07_...
1,county_tigerline_parquet,True,120.3448,E:\NetworkOptimization\pythonProject1\Data\07_...
2,county_master,True,0.2450,E:\NetworkOptimization\pythonProject1\Data\07_...
3,county_boundaries_master,True,16.1476,E:\NetworkOptimization\pythonProject1\Data\07_...
4,east_plus_gulf_counties,True,10.6237,E:\NetworkOptimization\pythonProject1\Data\07_...
5,state_summary,True,0.0035,E:\NetworkOptimization\pythonProject1\Data\07_...


,county_fips,statefp,countyfp,county_name,county_namelsad,state_name,centroid_lon,centroid_lat,rep_lon,rep_lat,is_conus,east_plus_gulf_flag,ALAND,AWATER,LSAD
0,01069,01,069,Houston,Houston County,Alabama,-85.302389,31.153231,-85.272645,31.152398,True,True,1501742250,4795418,06
1,01023,01,023,Choctaw,Choctaw County,Alabama,-88.263310,32.019539,-88.281606,32.010584,True,True,2365900084,19114321,06
2,01113,01,113,Russell,Russell County,Alabama,-85.184930,32.288366,-85.166736,32.295889,True,True,1660653961,15562947,06
3,10005,10,005,Sussex,Sussex County,Delaware,-75.389997,38.660481,-75.388972,38.676234,True,True,2424590442,674129051,06
4,01071,01,071,Jackson,Jackson County,Alabama,-85.999459,34.779435,-86.028104,34.740055,True,True,2792044612,126334711,06
5,01089,01,089,Madison,Madison County,Alabama,-86.550195,34.762956,-86.571973,34.747496,True,True,2076166271,28756353,06
6,04015,04,015,Mohave,Mohave County,Arizona,-113.762094,35.697868,-114.004195,35.583321,True,False,34530024143,333399390,06
7,05017,05,017,Chicot,Chicot County,Arkansas,-91.293718,33.266773,-91.292729,33.281026,True,True,1649764401,140413175,06
8,05121,05,121,Randolph,Randolph County,Arkansas,-91.027564,36.341471,-91.004937,36.311523,True,True,1688445994,10370823,06
9,05073,05,073,Lafayette,Lafayette County,Arkansas,-93.607132,33.241426,-93.596919,33.245227,True,True,1371327391,43028604,06


county_master rows: 3235

East-Plus-Gulf counts:


east_plus_gulf_flag
True     2510
False     725
Name: count, dtype: int64


State summary:


,statefp,state_name,is_conus,n_counties,n_east_plus_gulf,min_centroid_lon,max_centroid_lon
0,1,Alabama,True,67,67,-88.263310,-85.184930
1,2,Alaska,False,30,0,-173.802927,-130.926860
2,4,Arizona,True,15,0,-113.982672,-109.239951
3,5,Arkansas,True,75,75,-94.274420,-90.052379
4,6,California,True,58,0,-123.897141,-115.365362
5,8,Colorado,True,64,0,-108.596937,-102.351834
6,9,Connecticut,True,9,9,-73.447669,-71.976785
7,10,Delaware,True,3,3,-75.652782,-75.389997
8,11,District of Columbia,True,1,1,-77.016278,-77.016278
9,12,Florida,True,67,67,-87.362691,-80.431525



East-Plus-Gulf GeoDataFrame rows: 2510
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "eas

In [4]:
# ============================================================
# FREIGHT-MNet Processing Step 2:
# NTAD raw Esri JSON chunks -> GeoParquet chunks
#
# Input:
#   E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\*\raw_esri_json_chunks_v2
#
# Output:
#   E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\*\parquet_chunks_v2
#
# This does NOT redownload anything.
# It converts existing ArcGIS/Esri JSON chunks into GeoParquet.
# It is resumable: already-converted chunks are skipped.
# ============================================================

from pathlib import Path
import json
import re
import traceback
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString, Polygon, MultiPolygon
from shapely.validation import make_valid
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Paths and config
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
NTAD_ROOT = DATA_ROOT / "03_ntad_geospatial"
MANIFEST_DIR = DATA_ROOT / "00_manifest"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_MANIFEST_PATH = MANIFEST_DIR / "ntad_geoparquet_processing_manifest_v2.csv"
GLOBAL_LOG_PATH = MANIFEST_DIR / "ntad_geoparquet_processing_log_v2.jsonl"

LAYERS = {
    "faf_regions": {
        "dir": NTAD_ROOT / "faf_regions",
        "description": "FAF5 regions / FAF zone polygons",
        "combine": True,
    },
    "faf_network_links": {
        "dir": NTAD_ROOT / "faf_network_links",
        "description": "FAF5 highway freight network links",
        "combine": False,
    },
    "faf_network_nodes": {
        "dir": NTAD_ROOT / "faf_network_nodes",
        "description": "FAF5 highway freight network nodes",
        "combine": False,
    },
    "narn_lines": {
        "dir": NTAD_ROOT / "narn_lines",
        "description": "North American Rail Network lines",
        "combine": False,
    },
    "narn_nodes": {
        "dir": NTAD_ROOT / "narn_nodes",
        "description": "North American Rail Network nodes",
        "combine": False,
    },
    "intermodal_rail_tofc_cofc": {
        "dir": NTAD_ROOT / "intermodal_rail_tofc_cofc",
        "description": "Intermodal Rail TOFC/COFC terminal points",
        "combine": True,
    },
}

RESUME = True

# If True, deduplicate records across chunks using the ArcGIS OBJECTID field.
# This is important because earlier failed/split downloads may have overlapping chunks.
DEDUP_BY_OBJECTID = True

# If a chunk contains only duplicate rows, write a .empty.json marker instead of a parquet file.
WRITE_EMPTY_MARKERS = True

print("=" * 100)
print("NTAD raw Esri JSON -> GeoParquet processing")
print("DATA_ROOT:", DATA_ROOT)
print("NTAD_ROOT:", NTAD_ROOT)
print("=" * 100)

# ------------------------------------------------------------
# 1. Logging
# ------------------------------------------------------------

def log_event(record: dict) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with GLOBAL_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_size_mb(path: Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 4)


# ------------------------------------------------------------
# 2. Geometry conversion helpers
# ------------------------------------------------------------

def close_ring(coords: List[List[float]]) -> List[Tuple[float, float]]:
    pts = []
    for item in coords:
        if len(item) < 2:
            continue
        x, y = item[0], item[1]
        pts.append((float(x), float(y)))

    if len(pts) >= 3 and pts[0] != pts[-1]:
        pts.append(pts[0])

    return pts


def ring_signed_area(ring: List[Tuple[float, float]]) -> float:
    if len(ring) < 4:
        return 0.0

    area = 0.0
    for i in range(len(ring) - 1):
        x1, y1 = ring[i]
        x2, y2 = ring[i + 1]
        area += x1 * y2 - x2 * y1
    return area / 2.0


def esri_polygon_to_shapely(rings_raw: List[List[List[float]]]):
    """
    Convert Esri polygon rings to Shapely geometry.
    Uses ring orientation and containment heuristics.
    """
    rings = []
    for r in rings_raw:
        ring = close_ring(r)
        if len(ring) >= 4:
            rings.append(ring)

    if not rings:
        return None

    outer_candidates = []
    hole_candidates = []

    # Esri convention often uses clockwise outer rings.
    # With the signed-area formula in lon/lat, clockwise tends to be negative.
    for ring in rings:
        area = ring_signed_area(ring)
        if area < 0:
            outer_candidates.append(ring)
        else:
            hole_candidates.append(ring)

    # Fallback if orientation does not separate rings.
    if not outer_candidates:
        outer_candidates = [rings[0]]
        hole_candidates = rings[1:]

    polygons = []

    for outer in outer_candidates:
        try:
            outer_poly = Polygon(outer)
            holes_for_outer = []

            for hole in hole_candidates:
                try:
                    if outer_poly.contains(Point(hole[0])):
                        holes_for_outer.append(hole)
                except Exception:
                    pass

            poly = Polygon(outer, holes=holes_for_outer)
            poly = make_valid(poly)

            if not poly.is_empty:
                if poly.geom_type == "Polygon":
                    polygons.append(poly)
                elif poly.geom_type == "MultiPolygon":
                    polygons.extend(list(poly.geoms))

        except Exception:
            continue

    if not polygons:
        try:
            poly = Polygon(rings[0], holes=rings[1:])
            poly = make_valid(poly)
            return poly if not poly.is_empty else None
        except Exception:
            return None

    if len(polygons) == 1:
        return polygons[0]

    mp = MultiPolygon(polygons)
    mp = make_valid(mp)
    return mp if not mp.is_empty else None


def esri_geometry_to_shapely(geom: Optional[Dict[str, Any]]):
    if geom is None:
        return None

    # Point
    if "x" in geom and "y" in geom:
        try:
            return Point(float(geom["x"]), float(geom["y"]))
        except Exception:
            return None

    # Polyline
    if "paths" in geom:
        lines = []
        for path in geom.get("paths", []):
            coords = []
            for item in path:
                if len(item) >= 2:
                    coords.append((float(item[0]), float(item[1])))
            if len(coords) >= 2:
                try:
                    lines.append(LineString(coords))
                except Exception:
                    pass

        if not lines:
            return None
        if len(lines) == 1:
            return lines[0]
        return MultiLineString(lines)

    # Polygon
    if "rings" in geom:
        return esri_polygon_to_shapely(geom.get("rings", []))

    return None


# ------------------------------------------------------------
# 3. Metadata / object-id helpers
# ------------------------------------------------------------

def load_layer_metadata(layer_dir: Path, layer_name: str) -> dict:
    candidates = [
        layer_dir / "metadata_v2" / f"{layer_name}_metadata_v2.json",
        layer_dir / "metadata" / f"{layer_name}_metadata.json",
    ]

    for p in candidates:
        if p.exists():
            return json.loads(p.read_text(encoding="utf-8"))

    return {}


def get_oid_field(metadata: dict) -> str:
    for key in ["objectIdField", "objectIdFieldName"]:
        if metadata.get(key):
            return metadata[key]

    for f in metadata.get("fields", []):
        if f.get("type") == "esriFieldTypeOID":
            return f["name"]

    return "OBJECTID"


def find_oid_column(df: pd.DataFrame, oid_field: str) -> Optional[str]:
    if oid_field in df.columns:
        return oid_field

    lower_map = {c.lower(): c for c in df.columns}
    if oid_field.lower() in lower_map:
        return lower_map[oid_field.lower()]

    for candidate in ["OBJECTID", "objectid", "ObjectId", "OID", "fid", "FID"]:
        if candidate in df.columns:
            return candidate
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


def load_expected_object_ids(layer_dir: Path, layer_name: str) -> set:
    candidates = [
        layer_dir / "metadata_v2" / f"{layer_name}_object_ids_v2.json",
        layer_dir / "metadata" / f"{layer_name}_object_ids.json",
    ]

    for p in candidates:
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding="utf-8"))
                return set(int(x) for x in data)
            except Exception:
                return set()

    return set()


# ------------------------------------------------------------
# 4. JSON chunk conversion
# ------------------------------------------------------------

def sanitize_attributes(attrs: Dict[str, Any]) -> Dict[str, Any]:
    clean = {}
    for k, v in attrs.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        else:
            clean[k] = json.dumps(v, ensure_ascii=False)
    return clean


def esri_json_chunk_to_gdf(json_path: Path) -> gpd.GeoDataFrame:
    """
    Converts one ArcGIS/Esri JSON chunk into GeoDataFrame.
    """
    data = json.loads(json_path.read_text(encoding="utf-8"))

    # Some failed ArcGIS responses may contain error only.
    if isinstance(data, dict) and "error" in data:
        raise RuntimeError(f"ArcGIS error JSON in {json_path.name}: {data['error']}")

    features = data.get("features", [])
    if not isinstance(features, list):
        raise RuntimeError(f"No valid features list in {json_path}")

    rows = []
    geoms = []

    for i, feat in enumerate(features):
        attrs = feat.get("attributes", {})
        attrs = sanitize_attributes(attrs)

        geom = esri_geometry_to_shapely(feat.get("geometry"))
        attrs["__source_json"] = json_path.name
        attrs["__source_row"] = i

        rows.append(attrs)
        geoms.append(geom)

    if not rows:
        return gpd.GeoDataFrame(columns=["geometry"], geometry=[], crs="EPSG:4326")

    df = pd.DataFrame(rows)
    gdf = gpd.GeoDataFrame(df, geometry=geoms, crs="EPSG:4326")
    return gdf


def parquet_valid(path: Path) -> bool:
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        gdf = gpd.read_parquet(path)
        return len(gdf) > 0 and "geometry" in gdf.columns
    except Exception:
        return False


def empty_marker_valid(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0


def read_existing_oids(parquet_dir: Path, oid_field: str) -> set:
    seen = set()
    parquet_files = sorted(parquet_dir.glob("*.parquet"))

    for p in tqdm(parquet_files, desc=f"Reading existing OIDs from {parquet_dir.name}", leave=False):
        try:
            # Read only OID column through pandas/pyarrow if possible.
            df = pd.read_parquet(p)
            oid_col = find_oid_column(df, oid_field)

            if oid_col is not None:
                vals = pd.to_numeric(df[oid_col], errors="coerce").dropna().astype("int64")
                seen.update(vals.tolist())
        except Exception:
            pass

    return seen


def convert_layer(layer_name: str, cfg: dict) -> dict:
    layer_dir = cfg["dir"]
    raw_dir = layer_dir / "raw_esri_json_chunks_v2"
    parquet_dir = layer_dir / "parquet_chunks_v2"
    metadata_dir = layer_dir / "metadata_v2"

    parquet_dir.mkdir(parents=True, exist_ok=True)
    metadata_dir.mkdir(parents=True, exist_ok=True)

    metadata = load_layer_metadata(layer_dir, layer_name)
    oid_field = get_oid_field(metadata)
    expected_oids = load_expected_object_ids(layer_dir, layer_name)

    if not raw_dir.exists():
        raise FileNotFoundError(f"Missing raw_esri_json_chunks_v2 directory: {raw_dir}")

    json_files = sorted(raw_dir.glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"No JSON chunks found in: {raw_dir}")

    print("\n" + "=" * 120)
    print("LAYER:", layer_name)
    print(cfg["description"])
    print("raw_dir:", raw_dir)
    print("parquet_dir:", parquet_dir)
    print("n_json_chunks:", len(json_files))
    print("oid_field:", oid_field)
    print("expected_object_ids:", len(expected_oids) if expected_oids else "unknown")
    print("=" * 120)

    # Resume: load OIDs already saved in existing parquet chunks.
    seen_oids = set()
    if RESUME and DEDUP_BY_OBJECTID:
        seen_oids = read_existing_oids(parquet_dir, oid_field)
        if seen_oids:
            print(f"[resume] existing unique OIDs already in parquet: {len(seen_oids):,}")

    chunk_records = []
    failed = 0
    converted = 0
    skipped = 0
    empty = 0
    total_rows_written = 0
    total_rows_original = 0

    for json_path in tqdm(json_files, desc=f"Converting {layer_name}"):
        out_parquet = parquet_dir / f"{json_path.stem}.parquet"
        empty_marker = parquet_dir / f"{json_path.stem}.empty.json"

        rec = {
            "layer": layer_name,
            "json_path": str(json_path),
            "parquet_path": str(out_parquet),
            "empty_marker": str(empty_marker),
            "status": None,
            "n_rows_original": None,
            "n_rows_written": None,
            "n_duplicates_dropped": None,
            "error": None,
        }

        if RESUME and parquet_valid(out_parquet):
            try:
                gdf_existing = gpd.read_parquet(out_parquet)
                rec["status"] = "exists_valid"
                rec["n_rows_original"] = len(gdf_existing)
                rec["n_rows_written"] = len(gdf_existing)
                rec["parquet_size_mb"] = file_size_mb(out_parquet)
                skipped += 1
                total_rows_written += len(gdf_existing)
                chunk_records.append(rec)
                continue
            except Exception:
                pass

        if RESUME and empty_marker_valid(empty_marker):
            try:
                info = json.loads(empty_marker.read_text(encoding="utf-8"))
                rec.update(info)
                rec["status"] = "exists_empty_marker"
                skipped += 1
                empty += 1
                chunk_records.append(rec)
                continue
            except Exception:
                pass

        try:
            gdf = esri_json_chunk_to_gdf(json_path)
            n_original = len(gdf)
            total_rows_original += n_original

            if n_original == 0:
                rec["status"] = "empty_json_features"
                rec["n_rows_original"] = 0
                rec["n_rows_written"] = 0

                if WRITE_EMPTY_MARKERS:
                    empty_marker.write_text(json.dumps(rec, indent=2), encoding="utf-8")

                empty += 1
                chunk_records.append(rec)
                continue

            oid_col = find_oid_column(gdf, oid_field)

            duplicates_dropped = 0

            if DEDUP_BY_OBJECTID and oid_col is not None:
                oid_values = pd.to_numeric(gdf[oid_col], errors="coerce")

                keep_mask = []
                for val in oid_values:
                    if pd.isna(val):
                        # Keep rows with missing OID.
                        keep_mask.append(True)
                        continue

                    oid_int = int(val)
                    if oid_int in seen_oids:
                        keep_mask.append(False)
                    else:
                        keep_mask.append(True)
                        seen_oids.add(oid_int)

                keep_mask = pd.Series(keep_mask, index=gdf.index)
                duplicates_dropped = int((~keep_mask).sum())
                gdf = gdf.loc[keep_mask].copy()

            if len(gdf) == 0:
                rec["status"] = "all_duplicate_after_dedup"
                rec["n_rows_original"] = n_original
                rec["n_rows_written"] = 0
                rec["n_duplicates_dropped"] = duplicates_dropped

                if WRITE_EMPTY_MARKERS:
                    empty_marker.write_text(json.dumps(rec, indent=2), encoding="utf-8")

                empty += 1
                chunk_records.append(rec)
                continue

            # Make invalid geometries valid where possible.
            try:
                gdf["geometry"] = gdf["geometry"].apply(lambda x: make_valid(x) if x is not None and not x.is_valid else x)
            except Exception:
                pass

            gdf.to_parquet(out_parquet, index=False)

            rec["status"] = "converted"
            rec["n_rows_original"] = n_original
            rec["n_rows_written"] = len(gdf)
            rec["n_duplicates_dropped"] = duplicates_dropped
            rec["parquet_size_mb"] = file_size_mb(out_parquet)

            converted += 1
            total_rows_written += len(gdf)

        except Exception as e:
            rec["status"] = "failed"
            rec["error"] = repr(e)
            rec["traceback"] = traceback.format_exc()[:4000]
            failed += 1

        chunk_records.append(rec)

        # Save per-layer manifest periodically.
        if len(chunk_records) % 200 == 0:
            pd.DataFrame(chunk_records).to_csv(
                layer_dir / f"{layer_name}_geoparquet_chunk_manifest_v2.csv",
                index=False,
                encoding="utf-8-sig",
            )

    # Save chunk manifest.
    chunk_manifest = pd.DataFrame(chunk_records)
    chunk_manifest_path = layer_dir / f"{layer_name}_geoparquet_chunk_manifest_v2.csv"
    chunk_manifest.to_csv(chunk_manifest_path, index=False, encoding="utf-8-sig")

    # Count actual parquet chunks.
    parquet_files = sorted(parquet_dir.glob("*.parquet"))
    actual_rows = 0
    geom_types = {}

    for p in tqdm(parquet_files, desc=f"Counting parquet rows {layer_name}", leave=False):
        try:
            gdf = gpd.read_parquet(p)
            actual_rows += len(gdf)

            if "geometry" in gdf.columns and len(gdf) > 0:
                vc = gdf.geometry.geom_type.value_counts()
                for k, v in vc.items():
                    geom_types[k] = geom_types.get(k, 0) + int(v)

        except Exception:
            pass

    # Optionally combine small layers.
    combined_info = {}
    if cfg.get("combine", False):
        try:
            gdfs = [gpd.read_parquet(p) for p in parquet_files]
            if gdfs:
                combined = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs="EPSG:4326")

                # Final dedup just in case.
                oid_col = find_oid_column(combined, oid_field)
                if DEDUP_BY_OBJECTID and oid_col is not None:
                    before = len(combined)
                    combined[oid_col] = pd.to_numeric(combined[oid_col], errors="coerce")
                    combined = combined.drop_duplicates(subset=[oid_col], keep="first")
                    after = len(combined)
                    combined_info["combined_duplicates_dropped"] = before - after

                combined_parquet = layer_dir / f"{layer_name}_v2.parquet"
                combined_geojson = layer_dir / f"{layer_name}_v2.geojson"

                combined.to_parquet(combined_parquet, index=False)
                combined.to_file(combined_geojson, driver="GeoJSON")

                combined_info.update({
                    "combined_status": "created",
                    "combined_rows": len(combined),
                    "combined_parquet_path": str(combined_parquet),
                    "combined_geojson_path": str(combined_geojson),
                    "combined_parquet_size_mb": file_size_mb(combined_parquet),
                    "combined_geojson_size_mb": file_size_mb(combined_geojson),
                })
            else:
                combined_info["combined_status"] = "no_parquet_files"

        except Exception as e:
            combined_info["combined_status"] = "failed"
            combined_info["combined_error"] = repr(e)

    expected_count = len(expected_oids) if expected_oids else None

    if expected_count is not None:
        status = "ok" if actual_rows == expected_count else "check_count_mismatch"
    else:
        status = "ok" if actual_rows > 0 else "failed_no_rows"

    summary = {
        "layer": layer_name,
        "description": cfg["description"],
        "status": status,
        "layer_dir": str(layer_dir),
        "raw_dir": str(raw_dir),
        "parquet_dir": str(parquet_dir),
        "chunk_manifest_path": str(chunk_manifest_path),
        "n_json_chunks": len(json_files),
        "n_parquet_chunks": len(parquet_files),
        "n_empty_markers": len(list(parquet_dir.glob("*.empty.json"))),
        "converted_chunks_this_run": converted,
        "skipped_chunks_this_run": skipped,
        "failed_chunks_this_run": failed,
        "expected_object_ids": expected_count,
        "actual_rows_from_parquet": actual_rows,
        "seen_unique_oids": len(seen_oids) if DEDUP_BY_OBJECTID else None,
        "geometry_types": json.dumps(geom_types, ensure_ascii=False),
        **combined_info,
    }

    summary_path = layer_dir / f"{layer_name}_geoparquet_summary_v2.json"
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    log_event(summary)

    print("\nSUMMARY:", layer_name)
    for k, v in summary.items():
        if k not in {"geometry_types"}:
            print(f"  {k}: {v}")
    print("  geometry_types:", geom_types)

    return summary


# ------------------------------------------------------------
# 5. Run all layers
# ------------------------------------------------------------

summaries = []

for layer_name, cfg in LAYERS.items():
    try:
        summary = convert_layer(layer_name, cfg)
        summaries.append(summary)
    except Exception as e:
        err = {
            "layer": layer_name,
            "description": cfg.get("description"),
            "status": "failed",
            "error": repr(e),
            "traceback": traceback.format_exc()[:5000],
        }
        summaries.append(err)
        log_event(err)
        print("\nFAILED LAYER:", layer_name)
        print(repr(e))
        print(traceback.format_exc())

global_manifest = pd.DataFrame(summaries)
global_manifest.to_csv(GLOBAL_MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\n" + "=" * 100)
print("NTAD GEOPARQUET PROCESSING COMPLETE")
print("Global manifest:", GLOBAL_MANIFEST_PATH)
print("Global log:", GLOBAL_LOG_PATH)
print("=" * 100)

display_cols = [
    "layer",
    "status",
    "expected_object_ids",
    "actual_rows_from_parquet",
    "seen_unique_oids",
    "n_json_chunks",
    "n_parquet_chunks",
    "n_empty_markers",
    "failed_chunks_this_run",
    "combined_status",
    "combined_rows",
    "parquet_dir",
]
display_cols = [c for c in display_cols if c in global_manifest.columns]
display(global_manifest[display_cols])

NTAD raw Esri JSON -> GeoParquet processing
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
NTAD_ROOT: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial

LAYER: faf_regions
FAF5 regions / FAF zone polygons
raw_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\raw_esri_json_chunks_v2
parquet_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\parquet_chunks_v2
n_json_chunks: 3
oid_field: OBJECTID
expected_object_ids: 132


Reading existing OIDs from parquet_chunks_v2: 0it [00:00, ?it/s]

Converting faf_regions:   0%|          | 0/3 [00:00<?, ?it/s]

Counting parquet rows faf_regions:   0%|          | 0/1 [00:00<?, ?it/s]


SUMMARY: faf_regions
  layer: faf_regions
  description: FAF5 regions / FAF zone polygons
  status: ok
  layer_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions
  raw_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\raw_esri_json_chunks_v2
  parquet_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\parquet_chunks_v2
  chunk_manifest_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_geoparquet_chunk_manifest_v2.csv
  n_json_chunks: 3
  n_parquet_chunks: 1
  n_empty_markers: 2
  converted_chunks_this_run: 1
  skipped_chunks_this_run: 0
  failed_chunks_this_run: 0
  expected_object_ids: 132
  actual_rows_from_parquet: 132
  seen_unique_oids: 132
  combined_duplicates_dropped: 0
  combined_status: created
  combined_rows: 132
  combined_parquet_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_v2.parquet
  combined_geoj

...

Reading existing OIDs from parquet_chunks_v2: 0it [00:00, ?it/s]

Converting intermodal_rail_tofc_cofc:   0%|          | 0/4 [00:00<?, ?it/s]

Counting parquet rows intermodal_rail_tofc_cofc:   0%|          | 0/1 [00:00<?, ?it/s]


SUMMARY: intermodal_rail_tofc_cofc
  layer: intermodal_rail_tofc_cofc
  description: Intermodal Rail TOFC/COFC terminal points
  status: ok
  layer_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc
  raw_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\raw_esri_json_chunks_v2
  parquet_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\parquet_chunks_v2
  chunk_manifest_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\intermodal_rail_tofc_cofc_geoparquet_chunk_manifest_v2.csv
  n_json_chunks: 4
  n_parquet_chunks: 1
  n_empty_markers: 3
  converted_chunks_this_run: 1
  skipped_chunks_this_run: 0
  failed_chunks_this_run: 0
  expected_object_ids: 241
  actual_rows_from_parquet: 241
  seen_unique_oids: 241
  combined_duplicates_dropped: 0
  combined_status: created
  combined_rows: 241
  combined_parquet_path: E:\Ne

,layer,status,expected_object_ids,actual_rows_from_parquet,seen_unique_oids,n_json_chunks,n_parquet_chunks,n_empty_markers,failed_chunks_this_run,combined_status,combined_rows,parquet_dir
0,faf_regions,ok,132,132,132,3,1,2,0,created,132.0,E:\NetworkOptimization\pythonProject1\Data\03_...
1,faf_network_links,ok,487394,487394,487394,5362,488,4874,0,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\03_...
2,faf_network_nodes,ok,974788,974788,974788,10723,975,9748,0,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\03_...
3,narn_lines,ok,302743,302743,302743,3331,303,3028,0,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\03_...
4,narn_nodes,ok,250178,250178,250178,2753,251,2502,0,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\03_...
5,intermodal_rail_tofc_cofc,ok,241,241,241,4,1,3,0,created,241.0,E:\NetworkOptimization\pythonProject1\Data\03_...


In [5]:
# ============================================================
# FREIGHT-MNet Processing Step 3:
# Build county FIPS -> FAF zone crosswalk
#
# Inputs:
#   1. Census county boundaries:
#      E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\processed\county_boundaries_master_2024.parquet
#
#   2. FAF region polygons:
#      E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_v2.parquet
#
# Outputs:
#   E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf
#     - county_to_faf.parquet
#     - county_to_faf.csv
#     - county_to_faf_candidates.parquet
#     - east_plus_gulf_faf_zones.parquet
#     - east_plus_gulf_faf_zones.csv
#     - faf_regions_with_east_plus_gulf_flags.parquet
#     - county_to_faf_summary.json
#
# Method:
#   Main assignment = max-area spatial intersection between county polygon and FAF region polygon.
#   Fallback = representative-point spatial join / nearest FAF region if needed.
# ============================================================

from pathlib import Path
import json
import traceback

import pandas as pd
import geopandas as gpd
from shapely.validation import make_valid
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")

COUNTY_PATH = (
    DATA_ROOT
    / "07_census_boundaries"
    / "counties_2024"
    / "processed"
    / "county_boundaries_master_2024.parquet"
)

FAF_REGIONS_PATH = (
    DATA_ROOT
    / "03_ntad_geospatial"
    / "faf_regions"
    / "faf_regions_v2.parquet"
)

OUT_DIR = DATA_ROOT / "09_crosswalks" / "county_to_faf"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = DATA_ROOT / "00_manifest"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUT_DIR / "county_to_faf_summary.json"
CROSSWALK_PARQUET = OUT_DIR / "county_to_faf.parquet"
CROSSWALK_CSV = OUT_DIR / "county_to_faf.csv"
CANDIDATES_PARQUET = OUT_DIR / "county_to_faf_candidates.parquet"
EAST_FAF_PARQUET = OUT_DIR / "east_plus_gulf_faf_zones.parquet"
EAST_FAF_CSV = OUT_DIR / "east_plus_gulf_faf_zones.csv"
FAF_FLAGS_PARQUET = OUT_DIR / "faf_regions_with_east_plus_gulf_flags.parquet"
FAF_FLAGS_GEOJSON = OUT_DIR / "faf_regions_with_east_plus_gulf_flags.geojson"

EAST_PLUS_GULF_LONGITUDE_THRESHOLD = -100.0

print("=" * 100)
print("Build county FIPS -> FAF zone crosswalk")
print("COUNTY_PATH:", COUNTY_PATH)
print("FAF_REGIONS_PATH:", FAF_REGIONS_PATH)
print("OUT_DIR:", OUT_DIR)
print("=" * 100)

if not COUNTY_PATH.exists():
    raise FileNotFoundError(f"Missing county boundaries file: {COUNTY_PATH}")

if not FAF_REGIONS_PATH.exists():
    raise FileNotFoundError(f"Missing FAF regions file: {FAF_REGIONS_PATH}")

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def normalize_colname(c: str) -> str:
    return "".join(ch.lower() for ch in str(c) if ch.isalnum())


def detect_faf_id_column(faf: gpd.GeoDataFrame) -> str:
    """
    Detect the FAF zone ID column robustly.
    Expected possibilities include:
      FAFZONE, FAF_ZONE, FAF_Zone, FAF5, FAF5_ZONE, etc.
    """
    columns = list(faf.columns)
    norm_map = {normalize_colname(c): c for c in columns}

    exact_candidates = [
        "fafzone",
        "faf_zone",
        "faf5zone",
        "faf5_zone",
        "fafzoneid",
        "faf_zone_id",
        "zoneid",
        "zone",
    ]

    for cand in exact_candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]

    # Fuzzy: contains both faf and zone.
    for c in columns:
        nc = normalize_colname(c)
        if "faf" in nc and "zone" in nc and c != "geometry":
            return c

    # Fuzzy: contains faf and appears numeric.
    for c in columns:
        nc = normalize_colname(c)
        if "faf" in nc and c != "geometry":
            try:
                s = pd.to_numeric(faf[c], errors="coerce")
                if s.notna().sum() > 0:
                    return c
            except Exception:
                pass

    # Fallback to OBJECTID only if nothing else exists.
    for fallback in ["OBJECTID", "objectid", "ObjectId", "FID"]:
        if fallback in columns:
            print(f"WARNING: Could not detect FAF zone ID. Falling back to {fallback}.")
            return fallback

    raise RuntimeError(f"Could not detect FAF zone ID column. FAF columns: {columns}")


def detect_faf_name_column(faf: gpd.GeoDataFrame) -> str | None:
    columns = list(faf.columns)
    norm_map = {normalize_colname(c): c for c in columns}

    candidates = [
        "fafzonename",
        "faf_zone_name",
        "fafname",
        "zone_name",
        "zonename",
        "name",
        "description",
        "desc",
    ]

    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]

    # Fuzzy: any string-ish column containing name or desc.
    for c in columns:
        if c == "geometry":
            continue
        nc = normalize_colname(c)
        if "name" in nc or "desc" in nc:
            return c

    return None


def detect_county_columns(counties: gpd.GeoDataFrame) -> dict:
    cols = list(counties.columns)

    required = {
        "county_fips": ["county_fips", "GEOID", "geoid"],
        "statefp": ["statefp", "STATEFP"],
        "countyfp": ["countyfp", "COUNTYFP"],
        "county_name": ["county_name", "NAME"],
        "county_namelsad": ["county_namelsad", "NAMELSAD"],
        "east_plus_gulf_flag": ["east_plus_gulf_flag"],
        "centroid_lon": ["centroid_lon"],
        "centroid_lat": ["centroid_lat"],
        "rep_lon": ["rep_lon"],
        "rep_lat": ["rep_lat"],
    }

    found = {}

    for out_name, candidates in required.items():
        for c in candidates:
            if c in cols:
                found[out_name] = c
                break

    if "county_fips" not in found:
        raise RuntimeError(f"Could not detect county FIPS column. County columns: {cols}")

    return found


def safe_make_valid(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()

    def fix_geom(x):
        if x is None:
            return None
        try:
            if x.is_empty:
                return x
            if not x.is_valid:
                return make_valid(x)
            return x
        except Exception:
            return x

    gdf["geometry"] = gdf.geometry.apply(fix_geom)
    return gdf


# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

counties = gpd.read_parquet(COUNTY_PATH)
faf = gpd.read_parquet(FAF_REGIONS_PATH)

print("\nLoaded:")
print("  counties rows:", len(counties))
print("  counties CRS:", counties.crs)
print("  FAF regions rows:", len(faf))
print("  FAF regions CRS:", faf.crs)

if counties.crs is None:
    counties = counties.set_crs("EPSG:4326")
else:
    counties = counties.to_crs("EPSG:4326")

if faf.crs is None:
    faf = faf.set_crs("EPSG:4326")
else:
    faf = faf.to_crs("EPSG:4326")

counties = safe_make_valid(counties)
faf = safe_make_valid(faf)

county_cols = detect_county_columns(counties)
faf_zone_col = detect_faf_id_column(faf)
faf_name_col = detect_faf_name_column(faf)

print("\nDetected columns:")
print("  county columns:", county_cols)
print("  faf_zone_col:", faf_zone_col)
print("  faf_name_col:", faf_name_col)

# Standardize county identifiers.
counties["county_fips_std"] = counties[county_cols["county_fips"]].astype(str).str.zfill(5)

if "statefp" in county_cols:
    counties["statefp_std"] = counties[county_cols["statefp"]].astype(str).str.zfill(2)
else:
    counties["statefp_std"] = counties["county_fips_std"].str[:2]

if "countyfp" in county_cols:
    counties["countyfp_std"] = counties[county_cols["countyfp"]].astype(str).str.zfill(3)
else:
    counties["countyfp_std"] = counties["county_fips_std"].str[2:5]

# Standardize FAF ID.
faf["faf_zone_raw"] = faf[faf_zone_col]
faf["faf_zone"] = faf[faf_zone_col].astype(str).str.strip()

# If numeric-like, store numeric version too.
faf["faf_zone_numeric"] = pd.to_numeric(faf["faf_zone"], errors="coerce")

if faf_name_col is not None:
    faf["faf_zone_name"] = faf[faf_name_col].astype(str)
else:
    faf["faf_zone_name"] = faf["faf_zone"]

# ------------------------------------------------------------
# 3. Area-intersection assignment
# ------------------------------------------------------------

print("\nProjecting to EPSG:5070 for area-based assignment...")

counties_5070 = counties.to_crs("EPSG:5070").copy()
faf_5070 = faf.to_crs("EPSG:5070").copy()

# Keep only needed columns for sjoin.
county_keep = [
    "county_fips_std",
    "statefp_std",
    "countyfp_std",
    "geometry",
]

for src_key, col in county_cols.items():
    if col in counties_5070.columns and col not in county_keep:
        county_keep.append(col)

faf_keep = [
    "faf_zone",
    "faf_zone_numeric",
    "faf_zone_name",
    "geometry",
]

if faf_zone_col not in faf_keep and faf_zone_col in faf_5070.columns:
    faf_keep.append(faf_zone_col)

if faf_name_col is not None and faf_name_col not in faf_keep:
    faf_keep.append(faf_name_col)

county_small = counties_5070[county_keep].copy()
faf_small = faf_5070[faf_keep].copy()

county_small["county_area_m2"] = county_small.geometry.area

# Spatial candidate pairs.
print("Spatial join county polygons with FAF polygons...")
candidates = gpd.sjoin(
    county_small,
    faf_small,
    how="left",
    predicate="intersects"
)

print("Candidate pairs:", len(candidates))

# Compute actual intersection area.
faf_geom_by_index = faf_small.geometry.to_dict()

intersection_areas = []
for idx, row in tqdm(candidates.iterrows(), total=len(candidates), desc="Computing county-FAF overlap areas"):
    right_idx = row.get("index_right", None)

    if pd.isna(right_idx):
        intersection_areas.append(0.0)
        continue

    try:
        county_geom = row.geometry
        faf_geom = faf_geom_by_index[int(right_idx)]
        area = county_geom.intersection(faf_geom).area
    except Exception:
        area = 0.0

    intersection_areas.append(area)

candidates["overlap_area_m2"] = intersection_areas
candidates["overlap_share_of_county"] = candidates["overlap_area_m2"] / candidates["county_area_m2"].replace({0: pd.NA})
candidates["assignment_candidate_method"] = "area_intersection"

# Save candidate table without geometry to keep it light.
candidate_out = candidates.drop(columns="geometry").copy()
candidate_out.to_parquet(CANDIDATES_PARQUET, index=False)

# Pick max-area FAF region per county.
assigned = (
    candidates
    .sort_values(["county_fips_std", "overlap_area_m2"], ascending=[True, False])
    .groupby("county_fips_std", as_index=False)
    .head(1)
    .copy()
)

assigned["assignment_method"] = "max_area_intersection"

# ------------------------------------------------------------
# 4. Fallback for missing counties
# ------------------------------------------------------------

all_county_fips = set(counties["county_fips_std"])
assigned_fips = set(assigned["county_fips_std"].dropna().astype(str))
missing_fips = sorted(all_county_fips - assigned_fips)

print("\nAssigned by area:", len(assigned_fips))
print("Missing after area assignment:", len(missing_fips))

fallback_records = []

if missing_fips:
    print("Running fallback assignment using representative points / nearest FAF region...")

    missing = counties[counties["county_fips_std"].isin(missing_fips)].copy()

    # Build representative points. Prefer existing rep_lon/rep_lat.
    if "rep_lon" in county_cols and "rep_lat" in county_cols:
        pts = gpd.GeoDataFrame(
            missing.drop(columns="geometry").copy(),
            geometry=gpd.points_from_xy(
                missing[county_cols["rep_lon"]],
                missing[county_cols["rep_lat"]],
            ),
            crs="EPSG:4326",
        )
    else:
        pts = missing.copy()
        pts["geometry"] = pts.geometry.representative_point()

    pts_5070 = pts.to_crs("EPSG:5070")

    point_join = gpd.sjoin(
        pts_5070,
        faf_small,
        how="left",
        predicate="within"
    )

    still_missing = point_join[point_join["faf_zone"].isna()].copy()

    if len(still_missing) > 0:
        print("Some representative points did not fall inside FAF polygons. Using nearest fallback.")

        nearest = gpd.sjoin_nearest(
            still_missing.drop(columns=[c for c in ["index_right", "faf_zone", "faf_zone_numeric", "faf_zone_name"] if c in still_missing.columns]),
            faf_small,
            how="left",
            distance_col="nearest_distance_m"
        )

        point_join = pd.concat(
            [
                point_join[point_join["faf_zone"].notna()],
                nearest,
            ],
            ignore_index=True,
        )

    point_join["overlap_area_m2"] = pd.NA
    point_join["overlap_share_of_county"] = pd.NA
    point_join["assignment_method"] = "representative_point_or_nearest_fallback"

    assigned = pd.concat([assigned, point_join], ignore_index=True)

# ------------------------------------------------------------
# 5. Build final crosswalk
# ------------------------------------------------------------

# Merge back county attributes in original CRS.
county_attr_cols = [
    "county_fips_std",
    "statefp_std",
    "countyfp_std",
]

optional_cols = [
    county_cols.get("county_name"),
    county_cols.get("county_namelsad"),
    county_cols.get("centroid_lon"),
    county_cols.get("centroid_lat"),
    county_cols.get("rep_lon"),
    county_cols.get("rep_lat"),
    county_cols.get("east_plus_gulf_flag"),
]

for c in optional_cols:
    if c is not None and c in counties.columns and c not in county_attr_cols:
        county_attr_cols.append(c)

county_attrs = counties[county_attr_cols].drop_duplicates("county_fips_std").copy()

# Select assigned columns.
assign_cols = [
    "county_fips_std",
    "faf_zone",
    "faf_zone_numeric",
    "faf_zone_name",
    "overlap_area_m2",
    "overlap_share_of_county",
    "assignment_method",
]

assign_cols = [c for c in assign_cols if c in assigned.columns]

crosswalk = assigned[assign_cols].copy()
crosswalk = crosswalk.merge(county_attrs, on="county_fips_std", how="left")

# Rename project columns.
rename_map = {
    "county_fips_std": "county_fips",
    "statefp_std": "statefp",
    "countyfp_std": "countyfp",
}

# Rename original county columns if they exist.
if county_cols.get("county_name") in crosswalk.columns:
    rename_map[county_cols["county_name"]] = "county_name"
if county_cols.get("county_namelsad") in crosswalk.columns:
    rename_map[county_cols["county_namelsad"]] = "county_namelsad"
if county_cols.get("centroid_lon") in crosswalk.columns:
    rename_map[county_cols["centroid_lon"]] = "centroid_lon"
if county_cols.get("centroid_lat") in crosswalk.columns:
    rename_map[county_cols["centroid_lat"]] = "centroid_lat"
if county_cols.get("rep_lon") in crosswalk.columns:
    rename_map[county_cols["rep_lon"]] = "rep_lon"
if county_cols.get("rep_lat") in crosswalk.columns:
    rename_map[county_cols["rep_lat"]] = "rep_lat"
if county_cols.get("east_plus_gulf_flag") in crosswalk.columns:
    rename_map[county_cols["east_plus_gulf_flag"]] = "east_plus_gulf_county_flag"

crosswalk = crosswalk.rename(columns=rename_map)

# Clean types.
crosswalk["county_fips"] = crosswalk["county_fips"].astype(str).str.zfill(5)
crosswalk["statefp"] = crosswalk["statefp"].astype(str).str.zfill(2)
crosswalk["countyfp"] = crosswalk["countyfp"].astype(str).str.zfill(3)
crosswalk["faf_zone"] = crosswalk["faf_zone"].astype(str).str.strip()

if "east_plus_gulf_county_flag" not in crosswalk.columns:
    crosswalk["east_plus_gulf_county_flag"] = False

crosswalk["east_plus_gulf_county_flag"] = crosswalk["east_plus_gulf_county_flag"].astype(bool)

# Sort.
crosswalk = crosswalk.sort_values(["statefp", "county_fips"]).reset_index(drop=True)

# Sanity check: one FAF assignment per county.
dupes = crosswalk[crosswalk["county_fips"].duplicated(keep=False)]

if len(dupes) > 0:
    print("WARNING: duplicate county assignments found. Keeping first.")
    crosswalk = crosswalk.drop_duplicates("county_fips", keep="first").reset_index(drop=True)

# Save.
crosswalk.to_parquet(CROSSWALK_PARQUET, index=False)
crosswalk.to_csv(CROSSWALK_CSV, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 6. Build East-Plus-Gulf FAF zone summary
# ------------------------------------------------------------

faf_zone_summary = (
    crosswalk
    .groupby(["faf_zone", "faf_zone_name"], dropna=False)
    .agg(
        n_counties=("county_fips", "size"),
        n_east_plus_gulf_counties=("east_plus_gulf_county_flag", "sum"),
        min_county_centroid_lon=("centroid_lon", "min"),
        max_county_centroid_lon=("centroid_lon", "max"),
    )
    .reset_index()
)

faf_zone_summary["east_plus_gulf_faf_flag_any_county"] = (
    faf_zone_summary["n_east_plus_gulf_counties"] > 0
)

faf_zone_summary["east_plus_gulf_county_share"] = (
    faf_zone_summary["n_east_plus_gulf_counties"] / faf_zone_summary["n_counties"]
)

faf_zone_summary = faf_zone_summary.sort_values("faf_zone").reset_index(drop=True)

faf_zone_summary.to_parquet(EAST_FAF_PARQUET, index=False)
faf_zone_summary.to_csv(EAST_FAF_CSV, index=False, encoding="utf-8-sig")

# Attach summary flags to FAF region polygons.
faf_flags = faf.merge(
    faf_zone_summary[
        [
            "faf_zone",
            "n_counties",
            "n_east_plus_gulf_counties",
            "east_plus_gulf_faf_flag_any_county",
            "east_plus_gulf_county_share",
        ]
    ],
    on="faf_zone",
    how="left"
)

faf_flags["n_counties"] = faf_flags["n_counties"].fillna(0).astype(int)
faf_flags["n_east_plus_gulf_counties"] = faf_flags["n_east_plus_gulf_counties"].fillna(0).astype(int)
faf_flags["east_plus_gulf_faf_flag_any_county"] = faf_flags["east_plus_gulf_faf_flag_any_county"].fillna(False).astype(bool)
faf_flags["east_plus_gulf_county_share"] = faf_flags["east_plus_gulf_county_share"].fillna(0.0)

faf_flags.to_parquet(FAF_FLAGS_PARQUET, index=False)
faf_flags.to_file(FAF_FLAGS_GEOJSON, driver="GeoJSON")

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

summary = {
    "status": "ok",
    "county_input_path": str(COUNTY_PATH),
    "faf_regions_input_path": str(FAF_REGIONS_PATH),
    "county_to_faf_parquet": str(CROSSWALK_PARQUET),
    "county_to_faf_csv": str(CROSSWALK_CSV),
    "county_to_faf_candidates_parquet": str(CANDIDATES_PARQUET),
    "east_plus_gulf_faf_zones_parquet": str(EAST_FAF_PARQUET),
    "east_plus_gulf_faf_zones_csv": str(EAST_FAF_CSV),
    "faf_regions_with_flags_parquet": str(FAF_FLAGS_PARQUET),
    "faf_regions_with_flags_geojson": str(FAF_FLAGS_GEOJSON),
    "n_counties_total": int(len(crosswalk)),
    "n_unique_faf_zones_assigned": int(crosswalk["faf_zone"].nunique()),
    "n_east_plus_gulf_counties": int(crosswalk["east_plus_gulf_county_flag"].sum()),
    "n_east_plus_gulf_faf_zones_any_county": int(faf_zone_summary["east_plus_gulf_faf_flag_any_county"].sum()),
    "faf_zone_col_detected": faf_zone_col,
    "faf_name_col_detected": faf_name_col,
    "candidate_pairs": int(len(candidates)),
    "missing_after_area_assignment": int(len(missing_fips)),
    "longitude_threshold": EAST_PLUS_GULF_LONGITUDE_THRESHOLD,
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n" + "=" * 100)
print("COUNTY TO FAF CROSSWALK COMPLETE")
print("=" * 100)
print("county_to_faf:", CROSSWALK_PARQUET)
print("east_plus_gulf_faf_zones:", EAST_FAF_PARQUET)
print("summary:", SUMMARY_PATH)
print("\nCounts:")
for k, v in summary.items():
    if k.startswith("n_") or k in ["candidate_pairs", "missing_after_area_assignment"]:
        print(f"  {k}: {v}")

print("\nPreview county_to_faf:")
display(crosswalk.head(20))

print("\nPreview FAF zone summary:")
display(faf_zone_summary.head(20))

Build county FIPS -> FAF zone crosswalk
COUNTY_PATH: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\processed\county_boundaries_master_2024.parquet
FAF_REGIONS_PATH: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_v2.parquet
OUT_DIR: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf

Loaded:
  counties rows: 3235
  counties CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellips

Computing county-FAF overlap areas:   0%|          | 0/5802 [00:00<?, ?it/s]


Assigned by area: 3235
Missing after area assignment: 0

COUNTY TO FAF CROSSWALK COMPLETE
county_to_faf: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf.parquet
east_plus_gulf_faf_zones: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\east_plus_gulf_faf_zones.parquet
summary: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf_summary.json

Counts:
  n_counties_total: 3235
  n_unique_faf_zones_assigned: 133
  n_east_plus_gulf_counties: 2510
  n_east_plus_gulf_faf_zones_any_county: 104
  candidate_pairs: 5802
  missing_after_area_assignment: 0

Preview county_to_faf:


,county_fips,faf_zone,faf_zone_numeric,faf_zone_name,overlap_area_m2,overlap_share_of_county,assignment_method,statefp,countyfp,county_name,county_namelsad,centroid_lon,centroid_lat,rep_lon,rep_lat,east_plus_gulf_county_flag
0,01001,019,19.0,Remainder of Alabama,1.565252e+09,0.999954,max_area_intersection,01,001,Autauga,Autauga County,-86.642742,32.534915,-86.657503,32.511297,True
1,01003,012,12.0,"Mobile-Daphne-Fairhope, AL CFS Area",4.348760e+09,0.999547,max_area_intersection,01,003,Baldwin,Baldwin County,-87.722290,30.727099,-87.773825,30.759494,True
2,01005,019,19.0,Remainder of Alabama,2.342282e+09,0.999870,max_area_intersection,01,005,Barbour,Barbour County,-85.393439,31.869535,-85.395105,31.874560,True
3,01007,011,11.0,"Birmingham-Hoover-Talladega, AL CFS Area",1.621645e+09,0.999636,max_area_intersection,01,007,Bibb,Bibb County,-87.126503,32.998605,-87.096166,33.028324,True
4,01009,011,11.0,"Birmingham-Hoover-Talladega, AL CFS Area",1.685026e+09,0.999918,max_area_intersection,01,009,Blount,Blount County,-86.567544,33.980828,-86.527975,34.011475,True
5,01011,019,19.0,Remainder of Alabama,1.619183e+09,1.000000,max_area_intersection,01,011,Bullock,Bullock County,-85.715646,32.100555,-85.713520,32.097105,True
6,01013,019,19.0,Remainder of Alabama,2.014425e+09,1.000000,max_area_intersection,01,013,Butler,Butler County,-86.680286,31.752361,-86.676028,31.751035,True
7,01015,019,19.0,Remainder of Alabama,1.585609e+09,0.999910,max_area_intersection,01,015,Calhoun,Calhoun County,-85.826127,33.771404,-85.849441,33.773914,True
8,01017,019,19.0,Remainder of Alabama,1.561649e+09,0.999573,max_area_intersection,01,017,Chambers,Chambers County,-85.392008,32.914286,-85.391640,32.914703,True
9,01019,019,19.0,Remainder of Alabama,1.553557e+09,0.999890,max_area_intersection,01,019,Cherokee,Cherokee County,-85.603806,34.175778,-85.622103,34.243750,True



Preview FAF zone summary:


,faf_zone,faf_zone_name,n_counties,n_east_plus_gulf_counties,min_county_centroid_lon,max_county_centroid_lon,east_plus_gulf_faf_flag_any_county,east_plus_gulf_county_share
0,011,"Birmingham-Hoover-Talladega, AL CFS Area",11,11,-87.297210,-85.797517,True,1.0
1,012,"Mobile-Daphne-Fairhope, AL CFS Area",2,2,-88.205938,-87.722290,True,1.0
2,019,Remainder of Alabama,54,54,-88.263310,-85.184930,True,1.0
3,020,Remainder of Alaska,30,0,-173.802927,-130.926860,False,0.0
4,041,"Phoenix-Mesa-Scottsdale, AZ CFS Area",2,0,-112.493264,-111.344695,False,0.0
5,042,"Tucson-Nogales, AZ CFS Area",2,0,-111.789206,-110.846581,False,0.0
6,049,Remainder of Arizona,11,0,-113.982672,-109.239951,False,0.0
7,050,Remainder of Arkansas,75,75,-94.274420,-90.052379,True,1.0
8,061,"Los Angeles-Long Beach, CA CFS Area",5,0,-119.083380,-115.993876,False,0.0
9,062,"Sacramento-Roseville, CA CFS Area",7,0,-121.900539,-120.525051,False,0.0


In [6]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import json

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
OUT_DIR = DATA_ROOT / "09_crosswalks" / "county_to_faf"

paths = {
    "county_to_faf": OUT_DIR / "county_to_faf.parquet",
    "county_to_faf_csv": OUT_DIR / "county_to_faf.csv",
    "candidates": OUT_DIR / "county_to_faf_candidates.parquet",
    "east_plus_gulf_faf_zones": OUT_DIR / "east_plus_gulf_faf_zones.parquet",
    "faf_regions_with_flags": OUT_DIR / "faf_regions_with_east_plus_gulf_flags.parquet",
    "summary": OUT_DIR / "county_to_faf_summary.json",
}

rows = []
for name, p in paths.items():
    rows.append({
        "file": name,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024**2, 4) if p.exists() else 0,
        "path": str(p),
    })

check = pd.DataFrame(rows)
display(check)

crosswalk = pd.read_parquet(paths["county_to_faf"])
east_faf = pd.read_parquet(paths["east_plus_gulf_faf_zones"])

print("\ncounty_to_faf rows:", len(crosswalk))
print("unique counties:", crosswalk["county_fips"].nunique())
print("unique FAF zones:", crosswalk["faf_zone"].nunique())
print("east-plus-gulf counties:", int(crosswalk["east_plus_gulf_county_flag"].sum()))

print("\nAssignment methods:")
display(crosswalk["assignment_method"].value_counts(dropna=False))

print("\nOverlap share summary:")
display(crosswalk["overlap_share_of_county"].describe())

print("\nFAF zone summary:")
display(east_faf.head(30))

print("\nEast-Plus-Gulf FAF zones:")
display(east_faf[east_faf["east_plus_gulf_faf_flag_any_county"]].head(50))

summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
print("\nSummary JSON:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

# Basic assertions.
assert len(crosswalk) == crosswalk["county_fips"].nunique(), "Each county should appear once."
assert crosswalk["faf_zone"].notna().all(), "Every county should have a FAF zone."
assert crosswalk["east_plus_gulf_county_flag"].sum() > 0, "East-Plus-Gulf county count should be positive."

print("\nOK: county_to_faf crosswalk is ready.")

,file,exists,size_mb,path
0,county_to_faf,True,0.2437,E:\NetworkOptimization\pythonProject1\Data\09_...
1,county_to_faf_csv,True,0.6544,E:\NetworkOptimization\pythonProject1\Data\09_...
2,candidates,True,0.4328,E:\NetworkOptimization\pythonProject1\Data\09_...
3,east_plus_gulf_faf_zones,True,0.0122,E:\NetworkOptimization\pythonProject1\Data\09_...
4,faf_regions_with_flags,True,26.8149,E:\NetworkOptimization\pythonProject1\Data\09_...
5,summary,True,0.0016,E:\NetworkOptimization\pythonProject1\Data\09_...



county_to_faf rows: 3235
unique counties: 3235
unique FAF zones: 133
east-plus-gulf counties: 2510

Assignment methods:


assignment_method
max_area_intersection    3235
Name: count, dtype: int64


Overlap share summary:


count    3235.000000
mean        0.971586
std         0.165351
min         0.000000
25%         0.999803
50%         0.999955
75%         1.000000
max         1.000000
Name: overlap_share_of_county, dtype: float64


FAF zone summary:


,faf_zone,faf_zone_name,n_counties,n_east_plus_gulf_counties,min_county_centroid_lon,max_county_centroid_lon,east_plus_gulf_faf_flag_any_county,east_plus_gulf_county_share
0,011,"Birmingham-Hoover-Talladega, AL CFS Area",11,11,-87.297210,-85.797517,True,1.0
1,012,"Mobile-Daphne-Fairhope, AL CFS Area",2,2,-88.205938,-87.722290,True,1.0
2,019,Remainder of Alabama,54,54,-88.263310,-85.184930,True,1.0
3,020,Remainder of Alaska,30,0,-173.802927,-130.926860,False,0.0
4,041,"Phoenix-Mesa-Scottsdale, AZ CFS Area",2,0,-112.493264,-111.344695,False,0.0
5,042,"Tucson-Nogales, AZ CFS Area",2,0,-111.789206,-110.846581,False,0.0
6,049,Remainder of Arizona,11,0,-113.982672,-109.239951,False,0.0
7,050,Remainder of Arkansas,75,75,-94.274420,-90.052379,True,1.0
8,061,"Los Angeles-Long Beach, CA CFS Area",5,0,-119.083380,-115.993876,False,0.0
9,062,"Sacramento-Roseville, CA CFS Area",7,0,-121.900539,-120.525051,False,0.0



East-Plus-Gulf FAF zones:


,faf_zone,faf_zone_name,n_counties,n_east_plus_gulf_counties,min_county_centroid_lon,max_county_centroid_lon,east_plus_gulf_faf_flag_any_county,east_plus_gulf_county_share
0,011,"Birmingham-Hoover-Talladega, AL CFS Area",11,11,-87.297210,-85.797517,True,1.000000
1,012,"Mobile-Daphne-Fairhope, AL CFS Area",2,2,-88.205938,-87.722290,True,1.000000
2,019,Remainder of Alabama,54,54,-88.263310,-85.184930,True,1.000000
7,050,Remainder of Arkansas,75,75,-94.274420,-90.052379,True,1.000000
16,091,"Hartford-West Hartford-East Hartford, CT CFS ...",2,2,-72.572136,-72.507104,True,1.000000
17,092,"New York-Newark, NY-NJ-CT-PA CFS Area (CT Part)",5,5,-73.447669,-72.844574,True,1.000000
18,099,Remainder of Connecticut,2,2,-72.099882,-71.976785,True,1.000000
19,101,"Philadelphia-Reading-Camden, PA-NJ-DE-MD CFS ...",2,2,-75.652782,-75.568622,True,1.000000
20,109,Remainder of Delaware,1,1,-75.389997,-75.389997,True,1.000000
21,111,"Washington-Arlington-Alexandria, DC-VA-MD-WV ...",1,1,-77.016278,-77.016278,True,1.000000



Summary JSON:
{
  "status": "ok",
  "county_input_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\07_census_boundaries\\counties_2024\\processed\\county_boundaries_master_2024.parquet",
  "faf_regions_input_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\03_ntad_geospatial\\faf_regions\\faf_regions_v2.parquet",
  "county_to_faf_parquet": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\county_to_faf.parquet",
  "county_to_faf_csv": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\county_to_faf.csv",
  "county_to_faf_candidates_parquet": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\county_to_faf_candidates.parquet",
  "east_plus_gulf_faf_zones_parquet": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\east_plus_gulf_faf_zones.parquet",
  "east_plus_gulf_faf_zones_csv": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\east_plu

In [7]:
# ============================================================
# FREIGHT-MNet Repair Step:
# Fix county_to_faf crosswalk so FAF zone count matches FAF regions.
#
# Problem observed:
#   county_to_faf unique FAF zones = 133
#   FAF regions polygons = 132
#
# Likely cause:
#   Some Census county-equivalents outside FAF coverage were assigned string "nan".
#
# Output:
#   Data\09_crosswalks\county_to_faf\county_to_faf.parquet
#       Clean covered counties only.
#
#   Data\09_crosswalks\county_to_faf\county_to_faf_all_counties.parquet
#       All Census county-equivalents, including unmatched rows.
#
#   Data\09_crosswalks\county_to_faf\county_to_faf_unmatched.parquet
#       Counties not assigned to a real FAF polygon.
#
#   Data\09_crosswalks\county_to_faf\county_to_faf_repair_summary.json
# ============================================================

from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
from shapely.validation import make_valid
from tqdm.auto import tqdm

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")

COUNTY_PATH = (
    DATA_ROOT
    / "07_census_boundaries"
    / "counties_2024"
    / "processed"
    / "county_boundaries_master_2024.parquet"
)

FAF_REGIONS_PATH = (
    DATA_ROOT
    / "03_ntad_geospatial"
    / "faf_regions"
    / "faf_regions_v2.parquet"
)

OUT_DIR = DATA_ROOT / "09_crosswalks" / "county_to_faf"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CROSSWALK_CLEAN_PARQUET = OUT_DIR / "county_to_faf.parquet"
CROSSWALK_CLEAN_CSV = OUT_DIR / "county_to_faf.csv"

CROSSWALK_ALL_PARQUET = OUT_DIR / "county_to_faf_all_counties.parquet"
CROSSWALK_ALL_CSV = OUT_DIR / "county_to_faf_all_counties.csv"

UNMATCHED_PARQUET = OUT_DIR / "county_to_faf_unmatched.parquet"
UNMATCHED_CSV = OUT_DIR / "county_to_faf_unmatched.csv"

CANDIDATES_PARQUET = OUT_DIR / "county_to_faf_candidates_clean.parquet"

EAST_FAF_PARQUET = OUT_DIR / "east_plus_gulf_faf_zones.parquet"
EAST_FAF_CSV = OUT_DIR / "east_plus_gulf_faf_zones.csv"

FAF_FLAGS_PARQUET = OUT_DIR / "faf_regions_with_east_plus_gulf_flags.parquet"
FAF_FLAGS_GEOJSON = OUT_DIR / "faf_regions_with_east_plus_gulf_flags.geojson"

SUMMARY_PATH = OUT_DIR / "county_to_faf_repair_summary.json"

print("=" * 100)
print("Repair county_to_faf crosswalk")
print("COUNTY_PATH:", COUNTY_PATH)
print("FAF_REGIONS_PATH:", FAF_REGIONS_PATH)
print("OUT_DIR:", OUT_DIR)
print("=" * 100)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def normalize_colname(c: str) -> str:
    return "".join(ch.lower() for ch in str(c) if ch.isalnum())


def safe_make_valid(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()

    def fix_geom(x):
        if x is None:
            return None
        try:
            if x.is_empty:
                return x
            if not x.is_valid:
                return make_valid(x)
            return x
        except Exception:
            return x

    gdf["geometry"] = gdf.geometry.apply(fix_geom)
    return gdf


def detect_faf_id_column(faf: gpd.GeoDataFrame) -> str:
    cols = list(faf.columns)
    norm_map = {normalize_colname(c): c for c in cols}

    candidates = [
        "fafzone",
        "fafzoneid",
        "faf5zone",
        "faf5zoneid",
        "faf_zone",
        "faf_zone_id",
        "zoneid",
    ]

    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]

    for c in cols:
        nc = normalize_colname(c)
        if "faf" in nc and "zone" in nc and c != "geometry":
            return c

    raise RuntimeError(f"Could not detect FAF zone column. Columns: {cols}")


def detect_faf_name_column(faf: gpd.GeoDataFrame) -> str | None:
    cols = list(faf.columns)
    norm_map = {normalize_colname(c): c for c in cols}

    candidates = [
        "cfs17name",
        "fafzonename",
        "faf_zone_name",
        "zone_name",
        "name",
        "description",
    ]

    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]

    for c in cols:
        nc = normalize_colname(c)
        if "name" in nc or "desc" in nc:
            return c

    return None


def normalize_faf_zone_value(x):
    """
    Normalize FAF zone ID to string.
    Preserves leading zeros when present.
    Converts obvious missing-like values to pandas NA.
    """
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s.lower() in {"", "nan", "none", "null", "<na>"}:
        return pd.NA

    # If numeric-like and no leading zeros, pad to 3 digits.
    # If already like "011", keep as "011".
    try:
        if "." in s:
            f = float(s)
            if f.is_integer():
                s = str(int(f))
        if s.isdigit():
            s = s.zfill(3)
    except Exception:
        pass

    return s


# ------------------------------------------------------------
# Load inputs
# ------------------------------------------------------------

counties = gpd.read_parquet(COUNTY_PATH)
faf = gpd.read_parquet(FAF_REGIONS_PATH)

counties = counties.to_crs("EPSG:4326") if counties.crs else counties.set_crs("EPSG:4326")
faf = faf.to_crs("EPSG:4326") if faf.crs else faf.set_crs("EPSG:4326")

counties = safe_make_valid(counties)
faf = safe_make_valid(faf)

faf_zone_col = detect_faf_id_column(faf)
faf_name_col = detect_faf_name_column(faf)

print("Detected FAF zone column:", faf_zone_col)
print("Detected FAF name column:", faf_name_col)

# Standardize county columns.
if "county_fips" not in counties.columns:
    if "GEOID" in counties.columns:
        counties["county_fips"] = counties["GEOID"].astype(str).str.zfill(5)
    else:
        raise RuntimeError(f"Missing county_fips/GEOID. County columns: {list(counties.columns)}")

if "statefp" not in counties.columns:
    if "STATEFP" in counties.columns:
        counties["statefp"] = counties["STATEFP"].astype(str).str.zfill(2)
    else:
        counties["statefp"] = counties["county_fips"].astype(str).str[:2]

if "countyfp" not in counties.columns:
    if "COUNTYFP" in counties.columns:
        counties["countyfp"] = counties["COUNTYFP"].astype(str).str.zfill(3)
    else:
        counties["countyfp"] = counties["county_fips"].astype(str).str[2:5]

counties["county_fips"] = counties["county_fips"].astype(str).str.zfill(5)
counties["statefp"] = counties["statefp"].astype(str).str.zfill(2)
counties["countyfp"] = counties["countyfp"].astype(str).str.zfill(3)

if "county_name" not in counties.columns:
    if "NAME" in counties.columns:
        counties["county_name"] = counties["NAME"]
    else:
        counties["county_name"] = pd.NA

if "county_namelsad" not in counties.columns:
    if "NAMELSAD" in counties.columns:
        counties["county_namelsad"] = counties["NAMELSAD"]
    else:
        counties["county_namelsad"] = counties["county_name"]

if "east_plus_gulf_flag" not in counties.columns:
    if "centroid_lon" in counties.columns and "is_conus" in counties.columns:
        counties["east_plus_gulf_flag"] = counties["is_conus"] & (counties["centroid_lon"] >= -100.0)
    else:
        counties["east_plus_gulf_flag"] = False

# Standardize FAF.
faf["faf_zone"] = faf[faf_zone_col].apply(normalize_faf_zone_value)
faf["faf_zone_name"] = faf[faf_name_col].astype(str) if faf_name_col else faf["faf_zone"].astype(str)

valid_faf_zones = set(faf["faf_zone"].dropna().astype(str))
print("FAF region rows:", len(faf))
print("Valid unique FAF zones in FAF regions:", len(valid_faf_zones))

# ------------------------------------------------------------
# Area intersection assignment
# ------------------------------------------------------------

counties_5070 = counties.to_crs("EPSG:5070").copy()
faf_5070 = faf.to_crs("EPSG:5070").copy()

county_cols = [
    "county_fips",
    "statefp",
    "countyfp",
    "county_name",
    "county_namelsad",
    "centroid_lon",
    "centroid_lat",
    "rep_lon",
    "rep_lat",
    "is_conus",
    "east_plus_gulf_flag",
    "geometry",
]
county_cols = [c for c in county_cols if c in counties_5070.columns]

faf_cols = [
    "faf_zone",
    "faf_zone_name",
    "geometry",
]
faf_cols = [c for c in faf_cols if c in faf_5070.columns]

county_small = counties_5070[county_cols].copy()
faf_small = faf_5070[faf_cols].copy()

county_small["county_area_m2"] = county_small.geometry.area

print("\nSpatial join...")
candidates = gpd.sjoin(
    county_small,
    faf_small,
    how="left",
    predicate="intersects",
)

print("Candidate rows before cleaning:", len(candidates))

faf_geom_by_index = faf_small.geometry.to_dict()
overlap_areas = []

for idx, row in tqdm(candidates.iterrows(), total=len(candidates), desc="Computing overlap areas"):
    right_idx = row.get("index_right", None)

    if pd.isna(right_idx):
        overlap_areas.append(0.0)
        continue

    try:
        area = row.geometry.intersection(faf_geom_by_index[int(right_idx)]).area
    except Exception:
        area = 0.0

    overlap_areas.append(float(area))

candidates["overlap_area_m2"] = overlap_areas
candidates["overlap_share_of_county"] = (
    candidates["overlap_area_m2"] / candidates["county_area_m2"].replace({0: pd.NA})
)

# Normalize FAF values after join.
candidates["faf_zone"] = candidates["faf_zone"].apply(normalize_faf_zone_value)

# Critical cleaning:
# keep only real FAF zones and positive overlap.
valid_candidates = candidates[
    candidates["faf_zone"].notna()
    & candidates["faf_zone"].astype(str).isin(valid_faf_zones)
    & (candidates["overlap_area_m2"] > 0)
].copy()

print("Candidate rows after cleaning:", len(valid_candidates))

# Save candidates without geometry.
valid_candidates.drop(columns="geometry").to_parquet(CANDIDATES_PARQUET, index=False)

# Pick max-overlap FAF zone per county.
assigned = (
    valid_candidates
    .sort_values(["county_fips", "overlap_area_m2"], ascending=[True, False])
    .groupby("county_fips", as_index=False)
    .head(1)
    .copy()
)

assigned["assignment_method"] = "max_area_intersection_positive_overlap"

assigned_cols = [
    "county_fips",
    "statefp",
    "countyfp",
    "county_name",
    "county_namelsad",
    "centroid_lon",
    "centroid_lat",
    "rep_lon",
    "rep_lat",
    "is_conus",
    "east_plus_gulf_flag",
    "faf_zone",
    "faf_zone_name",
    "overlap_area_m2",
    "overlap_share_of_county",
    "assignment_method",
]
assigned_cols = [c for c in assigned_cols if c in assigned.columns]

covered = assigned[assigned_cols].copy()
covered = covered.drop_duplicates("county_fips", keep="first").reset_index(drop=True)

# Build all-counties table with missing info.
county_attrs = counties[
    [
        c for c in [
            "county_fips",
            "statefp",
            "countyfp",
            "county_name",
            "county_namelsad",
            "centroid_lon",
            "centroid_lat",
            "rep_lon",
            "rep_lat",
            "is_conus",
            "east_plus_gulf_flag",
        ]
        if c in counties.columns
    ]
].drop_duplicates("county_fips").copy()

all_counties = county_attrs.merge(
    covered[
        [
            "county_fips",
            "faf_zone",
            "faf_zone_name",
            "overlap_area_m2",
            "overlap_share_of_county",
            "assignment_method",
        ]
    ],
    on="county_fips",
    how="left",
)

all_counties["valid_faf_assignment"] = (
    all_counties["faf_zone"].notna()
    & all_counties["faf_zone"].astype(str).isin(valid_faf_zones)
)

unmatched = all_counties[~all_counties["valid_faf_assignment"]].copy()

# Rename flag for consistency with older downstream code.
covered = covered.rename(columns={"east_plus_gulf_flag": "east_plus_gulf_county_flag"})
all_counties = all_counties.rename(columns={"east_plus_gulf_flag": "east_plus_gulf_county_flag"})
unmatched = unmatched.rename(columns={"east_plus_gulf_flag": "east_plus_gulf_county_flag"})

covered["east_plus_gulf_county_flag"] = covered["east_plus_gulf_county_flag"].fillna(False).astype(bool)
all_counties["east_plus_gulf_county_flag"] = all_counties["east_plus_gulf_county_flag"].fillna(False).astype(bool)
unmatched["east_plus_gulf_county_flag"] = unmatched["east_plus_gulf_county_flag"].fillna(False).astype(bool)

# Sort.
covered = covered.sort_values(["statefp", "county_fips"]).reset_index(drop=True)
all_counties = all_counties.sort_values(["statefp", "county_fips"]).reset_index(drop=True)
unmatched = unmatched.sort_values(["statefp", "county_fips"]).reset_index(drop=True)

# Save.
covered.to_parquet(CROSSWALK_CLEAN_PARQUET, index=False)
covered.to_csv(CROSSWALK_CLEAN_CSV, index=False, encoding="utf-8-sig")

all_counties.to_parquet(CROSSWALK_ALL_PARQUET, index=False)
all_counties.to_csv(CROSSWALK_ALL_CSV, index=False, encoding="utf-8-sig")

unmatched.to_parquet(UNMATCHED_PARQUET, index=False)
unmatched.to_csv(UNMATCHED_CSV, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# FAF zone summary
# ------------------------------------------------------------

faf_zone_summary = (
    covered
    .groupby(["faf_zone", "faf_zone_name"], dropna=False)
    .agg(
        n_counties=("county_fips", "size"),
        n_east_plus_gulf_counties=("east_plus_gulf_county_flag", "sum"),
        min_county_centroid_lon=("centroid_lon", "min"),
        max_county_centroid_lon=("centroid_lon", "max"),
    )
    .reset_index()
)

faf_zone_summary["east_plus_gulf_faf_flag_any_county"] = (
    faf_zone_summary["n_east_plus_gulf_counties"] > 0
)

faf_zone_summary["east_plus_gulf_county_share"] = (
    faf_zone_summary["n_east_plus_gulf_counties"] / faf_zone_summary["n_counties"]
)

faf_zone_summary = faf_zone_summary.sort_values("faf_zone").reset_index(drop=True)

faf_zone_summary.to_parquet(EAST_FAF_PARQUET, index=False)
faf_zone_summary.to_csv(EAST_FAF_CSV, index=False, encoding="utf-8-sig")

faf_flags = faf.merge(
    faf_zone_summary[
        [
            "faf_zone",
            "n_counties",
            "n_east_plus_gulf_counties",
            "east_plus_gulf_faf_flag_any_county",
            "east_plus_gulf_county_share",
        ]
    ],
    on="faf_zone",
    how="left",
)

faf_flags["n_counties"] = faf_flags["n_counties"].fillna(0).astype(int)
faf_flags["n_east_plus_gulf_counties"] = faf_flags["n_east_plus_gulf_counties"].fillna(0).astype(int)
faf_flags["east_plus_gulf_faf_flag_any_county"] = faf_flags["east_plus_gulf_faf_flag_any_county"].fillna(False).astype(bool)
faf_flags["east_plus_gulf_county_share"] = faf_flags["east_plus_gulf_county_share"].fillna(0.0)

faf_flags.to_parquet(FAF_FLAGS_PARQUET, index=False)
faf_flags.to_file(FAF_FLAGS_GEOJSON, driver="GeoJSON")

# ------------------------------------------------------------
# Assertions and summary
# ------------------------------------------------------------

assert covered["county_fips"].is_unique, "Covered crosswalk should have one row per county."
assert covered["faf_zone"].notna().all(), "Covered crosswalk should have no missing FAF zones."
assert set(covered["faf_zone"].astype(str)).issubset(valid_faf_zones), "All assigned FAF zones must exist in FAF regions."
assert covered["overlap_area_m2"].gt(0).all(), "All covered county assignments must have positive overlap."

east_missing = all_counties[
    (all_counties["east_plus_gulf_county_flag"])
    & (~all_counties["valid_faf_assignment"])
]
assert len(east_missing) == 0, "All East-Plus-Gulf counties should have valid FAF assignments."

summary = {
    "status": "ok",
    "county_input_path": str(COUNTY_PATH),
    "faf_regions_input_path": str(FAF_REGIONS_PATH),
    "faf_zone_col_detected": faf_zone_col,
    "faf_name_col_detected": faf_name_col,
    "n_faf_region_rows": int(len(faf)),
    "n_unique_valid_faf_zones_from_regions": int(len(valid_faf_zones)),
    "n_census_counties_all": int(len(all_counties)),
    "n_counties_with_valid_faf_assignment": int(len(covered)),
    "n_unmatched_counties": int(len(unmatched)),
    "n_unique_faf_zones_assigned": int(covered["faf_zone"].nunique()),
    "n_east_plus_gulf_counties": int(covered["east_plus_gulf_county_flag"].sum()),
    "n_east_plus_gulf_faf_zones_any_county": int(faf_zone_summary["east_plus_gulf_faf_flag_any_county"].sum()),
    "candidate_pairs_raw": int(len(candidates)),
    "candidate_pairs_valid_positive_overlap": int(len(valid_candidates)),
    "county_to_faf_clean_parquet": str(CROSSWALK_CLEAN_PARQUET),
    "county_to_faf_all_counties_parquet": str(CROSSWALK_ALL_PARQUET),
    "county_to_faf_unmatched_parquet": str(UNMATCHED_PARQUET),
    "east_plus_gulf_faf_zones_parquet": str(EAST_FAF_PARQUET),
    "faf_regions_with_flags_parquet": str(FAF_FLAGS_PARQUET),
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n" + "=" * 100)
print("COUNTY_TO_FAF REPAIR COMPLETE")
print("=" * 100)
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nCovered crosswalk preview:")
display(covered.head(20))

print("\nUnmatched counties preview:")
display(unmatched.head(50))

print("\nFAF zone summary preview:")
display(faf_zone_summary.head(30))

Repair county_to_faf crosswalk
COUNTY_PATH: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\processed\county_boundaries_master_2024.parquet
FAF_REGIONS_PATH: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_v2.parquet
OUT_DIR: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf
Detected FAF zone column: FAF_Zone
Detected FAF name column: CFS17_NAME
FAF region rows: 132
Valid unique FAF zones in FAF regions: 132

Spatial join...
Candidate rows before cleaning: 5802


Computing overlap areas:   0%|          | 0/5802 [00:00<?, ?it/s]

Candidate rows after cleaning: 5711

COUNTY_TO_FAF REPAIR COMPLETE
status: ok
county_input_path: E:\NetworkOptimization\pythonProject1\Data\07_census_boundaries\counties_2024\processed\county_boundaries_master_2024.parquet
faf_regions_input_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_v2.parquet
faf_zone_col_detected: FAF_Zone
faf_name_col_detected: CFS17_NAME
n_faf_region_rows: 132
n_unique_valid_faf_zones_from_regions: 132
n_census_counties_all: 3235
n_counties_with_valid_faf_assignment: 3144
n_unmatched_counties: 91
n_unique_faf_zones_assigned: 132
n_east_plus_gulf_counties: 2510
n_east_plus_gulf_faf_zones_any_county: 104
candidate_pairs_raw: 5802
candidate_pairs_valid_positive_overlap: 5711
county_to_faf_clean_parquet: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf.parquet
county_to_faf_all_counties_parquet: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf_all_co

,county_fips,statefp,countyfp,county_name,county_namelsad,centroid_lon,centroid_lat,rep_lon,rep_lat,is_conus,east_plus_gulf_county_flag,faf_zone,faf_zone_name,overlap_area_m2,overlap_share_of_county,assignment_method
0,01001,01,001,Autauga,Autauga County,-86.642742,32.534915,-86.657503,32.511297,True,True,019,Remainder of Alabama,1.565252e+09,0.999954,max_area_intersection_positive_overlap
1,01003,01,003,Baldwin,Baldwin County,-87.722290,30.727099,-87.773825,30.759494,True,True,012,"Mobile-Daphne-Fairhope, AL CFS Area",4.348760e+09,0.999547,max_area_intersection_positive_overlap
2,01005,01,005,Barbour,Barbour County,-85.393439,31.869535,-85.395105,31.874560,True,True,019,Remainder of Alabama,2.342282e+09,0.999870,max_area_intersection_positive_overlap
3,01007,01,007,Bibb,Bibb County,-87.126503,32.998605,-87.096166,33.028324,True,True,011,"Birmingham-Hoover-Talladega, AL CFS Area",1.621645e+09,0.999636,max_area_intersection_positive_overlap
4,01009,01,009,Blount,Blount County,-86.567544,33.980828,-86.527975,34.011475,True,True,011,"Birmingham-Hoover-Talladega, AL CFS Area",1.685026e+09,0.999918,max_area_intersection_positive_overlap
5,01011,01,011,Bullock,Bullock County,-85.715646,32.100555,-85.713520,32.097105,True,True,019,Remainder of Alabama,1.619183e+09,1.000000,max_area_intersection_positive_overlap
6,01013,01,013,Butler,Butler County,-86.680286,31.752361,-86.676028,31.751035,True,True,019,Remainder of Alabama,2.014425e+09,1.000000,max_area_intersection_positive_overlap
7,01015,01,015,Calhoun,Calhoun County,-85.826127,33.771404,-85.849441,33.773914,True,True,019,Remainder of Alabama,1.585609e+09,0.999910,max_area_intersection_positive_overlap
8,01017,01,017,Chambers,Chambers County,-85.392008,32.914286,-85.391640,32.914703,True,True,019,Remainder of Alabama,1.561649e+09,0.999573,max_area_intersection_positive_overlap
9,01019,01,019,Cherokee,Cherokee County,-85.603806,34.175778,-85.622103,34.243750,True,True,019,Remainder of Alabama,1.553557e+09,0.999890,max_area_intersection_positive_overlap



Unmatched counties preview:


,county_fips,statefp,countyfp,county_name,county_namelsad,centroid_lon,centroid_lat,rep_lon,rep_lat,is_conus,east_plus_gulf_county_flag,faf_zone,faf_zone_name,overlap_area_m2,overlap_share_of_county,assignment_method,valid_faf_assignment
0,60010,60,010,Eastern,Eastern District,-170.658744,-14.274324,-170.646561,-14.268633,False,False,NaN,NaN,NaN,NaN,NaN,False
1,60020,60,020,Manu'a,Manu'a District,-169.506017,-14.221707,-169.473145,-14.241720,False,False,NaN,NaN,NaN,NaN,NaN,False
2,60030,60,030,Rose Island,Rose Island,-168.146830,-14.543666,-168.144716,-14.547284,False,False,NaN,NaN,NaN,NaN,NaN,False
3,60040,60,040,Swains Island,Swains Island,-171.078174,-11.054967,-171.079444,-11.054818,False,False,NaN,NaN,NaN,NaN,NaN,False
4,60050,60,050,Western,Western District,-170.768190,-14.323515,-170.771804,-14.318822,False,False,NaN,NaN,NaN,NaN,NaN,False
5,66010,66,010,Guam,Guam,144.774163,13.443833,144.744106,13.377310,False,False,NaN,NaN,NaN,NaN,NaN,False
6,69085,69,085,Northern Islands,Northern Islands Municipality,145.692838,17.948740,145.769709,18.128801,False,False,NaN,NaN,NaN,NaN,NaN,False
7,69100,69,100,Rota,Rota Municipality,145.214490,14.157339,145.204572,14.151018,False,False,NaN,NaN,NaN,NaN,NaN,False
8,69110,69,110,Saipan,Saipan Municipality,145.753440,15.188774,145.752607,15.178923,False,False,NaN,NaN,NaN,NaN,NaN,False
9,69120,69,120,Tinian,Tinian Municipality,145.626893,15.003148,145.637146,15.026981,False,False,NaN,NaN,NaN,NaN,NaN,False



FAF zone summary preview:


,faf_zone,faf_zone_name,n_counties,n_east_plus_gulf_counties,min_county_centroid_lon,max_county_centroid_lon,east_plus_gulf_faf_flag_any_county,east_plus_gulf_county_share
0,011,"Birmingham-Hoover-Talladega, AL CFS Area",11,11,-87.297210,-85.797517,True,1.0
1,012,"Mobile-Daphne-Fairhope, AL CFS Area",2,2,-88.205938,-87.722290,True,1.0
2,019,Remainder of Alabama,54,54,-88.263310,-85.184930,True,1.0
3,020,Remainder of Alaska,30,0,-173.802927,-130.926860,False,0.0
4,041,"Phoenix-Mesa-Scottsdale, AZ CFS Area",2,0,-112.493264,-111.344695,False,0.0
5,042,"Tucson-Nogales, AZ CFS Area",2,0,-111.789206,-110.846581,False,0.0
6,049,Remainder of Arizona,11,0,-113.982672,-109.239951,False,0.0
7,050,Remainder of Arkansas,75,75,-94.274420,-90.052379,True,1.0
8,061,"Los Angeles-Long Beach, CA CFS Area",5,0,-119.083380,-115.993876,False,0.0
9,062,"Sacramento-Roseville, CA CFS Area",7,0,-121.900539,-120.525051,False,0.0


In [8]:
from pathlib import Path
import json
import pandas as pd

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
OUT_DIR = DATA_ROOT / "09_crosswalks" / "county_to_faf"

paths = {
    "county_to_faf_clean": OUT_DIR / "county_to_faf.parquet",
    "county_to_faf_all": OUT_DIR / "county_to_faf_all_counties.parquet",
    "county_to_faf_unmatched": OUT_DIR / "county_to_faf_unmatched.parquet",
    "east_plus_gulf_faf_zones": OUT_DIR / "east_plus_gulf_faf_zones.parquet",
    "summary": OUT_DIR / "county_to_faf_repair_summary.json",
}

for name, path in paths.items():
    print(name, path.exists(), path)

crosswalk = pd.read_parquet(paths["county_to_faf_clean"])
all_counties = pd.read_parquet(paths["county_to_faf_all"])
unmatched = pd.read_parquet(paths["county_to_faf_unmatched"])
east_faf = pd.read_parquet(paths["east_plus_gulf_faf_zones"])
summary = json.loads(paths["summary"].read_text(encoding="utf-8"))

print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nClean county_to_faf:")
print("rows:", len(crosswalk))
print("unique counties:", crosswalk["county_fips"].nunique())
print("unique FAF zones:", crosswalk["faf_zone"].nunique())
print("east-plus-gulf counties:", int(crosswalk["east_plus_gulf_county_flag"].sum()))
print("missing FAF values:", crosswalk["faf_zone"].isna().sum())
print("string nan FAF values:", (crosswalk["faf_zone"].astype(str).str.lower() == "nan").sum())

print("\nAll counties:")
print("rows:", len(all_counties))
print("valid assignments:", int(all_counties["valid_faf_assignment"].sum()))
print("unmatched:", len(unmatched))

print("\nUnmatched by state:")
display(
    unmatched
    .groupby(["statefp"], dropna=False)
    .size()
    .reset_index(name="n_unmatched")
    .sort_values("statefp")
)

print("\nAssignment methods:")
display(crosswalk["assignment_method"].value_counts(dropna=False))

print("\nOverlap share summary:")
display(crosswalk["overlap_share_of_county"].describe())

print("\nEast FAF zones:")
display(east_faf.head(50))

# Assertions.
assert crosswalk["county_fips"].is_unique
assert crosswalk["faf_zone"].notna().all()
assert (crosswalk["faf_zone"].astype(str).str.lower() != "nan").all()
assert crosswalk["overlap_area_m2"].gt(0).all()
assert crosswalk["faf_zone"].nunique() <= summary["n_unique_valid_faf_zones_from_regions"]
assert crosswalk["east_plus_gulf_county_flag"].sum() == 2510

print("\nOK: repaired county_to_faf crosswalk is clean.")

county_to_faf_clean True E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf.parquet
county_to_faf_all True E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf_all_counties.parquet
county_to_faf_unmatched True E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf_unmatched.parquet
east_plus_gulf_faf_zones True E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\east_plus_gulf_faf_zones.parquet
summary True E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf_repair_summary.json

Summary:
{
  "status": "ok",
  "county_input_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\07_census_boundaries\\counties_2024\\processed\\county_boundaries_master_2024.parquet",
  "faf_regions_input_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\03_ntad_geospatial\\faf_regions\\faf_regions_v2.parquet",
  "faf_zone_col_detected": "FAF_Zone",
  "faf_

,statefp,n_unmatched
0,60,5
1,66,1
2,69,4
3,72,78
4,78,3



Assignment methods:


assignment_method
max_area_intersection_positive_overlap    3144
Name: count, dtype: int64


Overlap share summary:


count    3144.000000
mean        0.999707
std         0.003225
min         0.865895
25%         0.999823
50%         0.999959
75%         1.000000
max         1.000000
Name: overlap_share_of_county, dtype: float64


East FAF zones:


,faf_zone,faf_zone_name,n_counties,n_east_plus_gulf_counties,min_county_centroid_lon,max_county_centroid_lon,east_plus_gulf_faf_flag_any_county,east_plus_gulf_county_share
0,011,"Birmingham-Hoover-Talladega, AL CFS Area",11,11,-87.297210,-85.797517,True,1.000000
1,012,"Mobile-Daphne-Fairhope, AL CFS Area",2,2,-88.205938,-87.722290,True,1.000000
2,019,Remainder of Alabama,54,54,-88.263310,-85.184930,True,1.000000
3,020,Remainder of Alaska,30,0,-173.802927,-130.926860,False,0.000000
4,041,"Phoenix-Mesa-Scottsdale, AZ CFS Area",2,0,-112.493264,-111.344695,False,0.000000
5,042,"Tucson-Nogales, AZ CFS Area",2,0,-111.789206,-110.846581,False,0.000000
6,049,Remainder of Arizona,11,0,-113.982672,-109.239951,False,0.000000
7,050,Remainder of Arkansas,75,75,-94.274420,-90.052379,True,1.000000
8,061,"Los Angeles-Long Beach, CA CFS Area",5,0,-119.083380,-115.993876,False,0.000000
9,062,"Sacramento-Roseville, CA CFS Area",7,0,-121.900539,-120.525051,False,0.000000



OK: repaired county_to_faf crosswalk is clean.
